# 🎬 YouTube Shorts ワンクリック自動生成

---

## ✏️ 毎回やること（セル1だけ変える）

| 設定項目 | 説明 |
|---|---|
| `YOUTUBE_API_KEY` | **初回だけ**入力。Google Cloud Console で取得（**無料**） |
| `CLAUDE_API_KEY` | **初回だけ**入力。console.anthropic.com で取得（有料・`sk-ant-`で始まる） |
| `PEXELS_API_KEY` | **初回だけ**入力。pexels.com/api で取得（**無料**） |
| `THEME` | **毎回**テーマを書き換える |
| `VIDEO_COUNT` | 生成する本数（1〜10） |

あとは「**ランタイム → すべてのセルを実行**」をクリックするだけ！

---

## 💰 Claude APIの費用目安
| 使い方 | 費用 |
|---|---|
| 1回10本生成 | 約5円 |
| 毎日10本 × 30日 | 約160円/月 |

---

## 🎨 画像スタイル
| スタイル名 | 見た目 | 向いているテーマ |
|---|---|---|
| `realistic` | 写真そのまま | 筋トレ・料理・ビジネス |
| `anime` | アニメ風 | 恋愛・感情・エンタメ |
| `manga` | 漫画風（白黒） | 怖い話・歴史・雑学 |
| `illustration` | イラスト風 | 子ども向け・ライフスタイル |

---

## ⏱ 目安時間
| 本数 | 目安 |
|---|---|
| 1本 | 約5〜10分 |
| 5本 | 約25〜40分 |
| 10本 | 約50〜80分 |

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  セル1: テーマと本数だけ変える（APIキーは初回のみ）  ║
# ╚══════════════════════════════════════════════════════╝

# ── ① APIキー（初回のみ。2回目以降は自動読み込み）──────
YOUTUBE_API_KEY = ''
CLAUDE_API_KEY  = ''
PEXELS_API_KEY  = ''
GOOGLE_API_KEY  = ''   # aistudio.google.com
ELEVENLABS_API_KEY = '' # elevenlabs.io（未設定時はgTTSを使用）

# ── ② テーマ（複数入力OK → AIが自動でチャンネル振り分け）──
# 例: ['ダイエット', '投資', '腸活', 'NISA']
THEMES = ['ダイエット']

# ── ③ 生成する動画の本数（テーマ1つあたり）────────────────
VIDEOS_PER_THEME = 1

# ── ④ 動画の長さ ─────────────────────────────────────────
DURATION = 45

# ── ⑤ 画像スタイル ───────────────────────────────────────
IMAGE_STYLE = 'realistic'

# ── ⑥ YouTube自動投稿設定 ──────────────────────────────
AUTO_UPLOAD    = True        # True: 生成後に自動投稿 / False: 生成のみ
UPLOAD_PRIVACY = 'public'    # 'public' / 'private' / 'unlisted'
SCHEDULE_POST  = True        # True: ランダム時間帯にスケジュール投稿

# ── ⑦ ランダム投稿ウィンドウ（JST）[時, 開始分, 終了分] ────
POST_WINDOWS = [
    (8,  0, 30),   # 8:00〜8:30
    (12, 0, 30),   # 12:00〜12:30
    (16, 0, 30),   # 16:00〜16:30
    (18, 0, 30),   # 18:00〜18:30
    (21, 0, 30),   # 21:00〜21:30
]

# ── ⑧ チャンネル設定（自動分類されますが手動で上書き可）──
# 空白のままにしておくと、テーマからAIが自動で判断します
# 例: FORCE_CHANNEL = 'yaseru'  # やせる習慣図鑑に強制
#   : FORCE_CHANNEL = 'okane'   # お金の教科書に強制
FORCE_CHANNEL = ''

# ── ⑨ Google Drive保存先フォルダID ────────────────────
DRIVE_FOLDER_ID = ''

# ── ⑩ BGMフォルダ（YouTube Audio Library等のMP3を入れる）─────────
# Google Drive の MyDrive/BGM/ に音楽ファイル(.mp3/.wav)を置くと自動で使用
# チャンネル別: BGM/yaseru/ と BGM/okane/ に分けると自動切替（任意）
# 空のままでもOK → 自動生成BGMを使用
BGM_FOLDER = '/content/drive/MyDrive/BGM'

# ── ⑪ チャンネルID（すでにシークレット登録済みなら空白でOK）──────────
# ColabシークレットのCHANNEL_ID_1（やせる習慣図鑑）・CHANNEL_ID_2（お金の教科書）
# を自動で読み込みます。ここは変更不要。
YASERU_CHANNEL_ID = ''   # ← 自動入力（シークレット: CHANNEL_ID_1）
OKANE_CHANNEL_ID  = ''   # ← 自動入力（シークレット: CHANNEL_ID_2）

# ── ⑫ その他設定 ──────────────────────────────────────
LOOP_STRUCTURE   = True
GEMINI_IMAGE_MODEL = ''
IMG_SWITCH_SEC   = 2.0
VIDEO_FORMAT     = 'standard'
CHANNEL_NAME     = ''
DIAGNOSIS_ITEMS  = 4
MASCOT_PROMPT    = ''
ASMR_TICK        = False

# ╔══════════════════════════════════════════════════════╗
# ║  ↑ 変えるのはここまで。以下は触らなくてOK            ║
# ╚══════════════════════════════════════════════════════╝

# ── APIキーをColabシークレットから自動取得 ───────────────
try:
    from google.colab import userdata as _ud
    def _clean(v): return ''.join(c for c in (v or '') if 0x21 <= ord(c) <= 0x7E)
    if not YOUTUBE_API_KEY.strip():    YOUTUBE_API_KEY    = _clean(_ud.get('YOUTUBE_API_KEY'))
    if not CLAUDE_API_KEY.strip():     CLAUDE_API_KEY     = _clean(_ud.get('CLAUDE_API_KEY'))
    if not PEXELS_API_KEY.strip():     PEXELS_API_KEY     = _clean(_ud.get('PEXELS_API_KEY'))
    if not GOOGLE_API_KEY.strip():     GOOGLE_API_KEY     = _clean(_ud.get('GOOGLE_API_KEY'))
    if not ELEVENLABS_API_KEY.strip():
        _el_raw = (_ud.get('ELEVENLABS_API_KEY') or '').strip()
        ELEVENLABS_API_KEY = _el_raw  # ElevenLabsキーは_cleanせずそのままstrip
    if not DRIVE_FOLDER_ID.strip():    DRIVE_FOLDER_ID    = _clean(_ud.get('DRIVE_FOLDER_ID'))
    # CHANNEL_ID_1 = やせる習慣図鑑 / CHANNEL_ID_2 = お金の教科書
    if not YASERU_CHANNEL_ID.strip():
        YASERU_CHANNEL_ID = _clean(_ud.get('CHANNEL_ID_1') or _ud.get('YASERU_CHANNEL_ID') or '')
    if not OKANE_CHANNEL_ID.strip():
        OKANE_CHANNEL_ID  = _clean(_ud.get('CHANNEL_ID_2') or _ud.get('OKANE_CHANNEL_ID') or '')
    _src = 'Colabシークレット'
except Exception as _e_secrets:
    print(f'  ⚠ Colabシークレット読み込み失敗: {_e_secrets}')
    _src = 'セル内の直接入力'

# ── テーマ後方互換（THEME単数 → THEMESリスト）────────────
if 'THEME' in dir() and not globals().get('THEMES'):
    THEMES = [THEME]
THEME = THEMES[0] if THEMES else 'ダイエット'  # Cell4/5互換用
VIDEO_COUNT = VIDEOS_PER_THEME  # Cell5互換用

# ── バリデーション ────────────────────────────────────────
import os
if not YOUTUBE_API_KEY.strip():
    raise ValueError('❌ YOUTUBE_API_KEY が未設定。Colabシークレットに登録してください。')
if not CLAUDE_API_KEY.strip():
    raise ValueError('❌ CLAUDE_API_KEY が未設定。Colabシークレットに登録してください。')
if not PEXELS_API_KEY.strip():
    print('⚠ PEXELS_API_KEY 未設定（Gemini画像生成を使用）')
os.makedirs('/content/output', exist_ok=True)

# ── client_secrets.json を3ステップで取得 ────────────────
_csj_path = '/content/client_secrets.json'

# Step 1: Colabシークレット（CLIENT_SECRETS_JSON）
if not os.path.exists(_csj_path):
    print('  [Step 1] Colabシークレットから client_secrets.json を取得中...')
    try:
        from google.colab import userdata as _ud_cs
        _csj = (_ud_cs.get('CLIENT_SECRETS_JSON') or '').strip()
        if _csj:
            import json as _json_cs
            _json_cs.loads(_csj)  # JSON構文チェック（不正なJSONを早期検出）
            with open(_csj_path, 'w') as _csf:
                _csf.write(_csj)
            print('  🔑 client_secrets.json をシークレットから自動作成しました')
        else:
            print('  ⚠ CLIENT_SECRETS_JSON シークレットが空です')
            print('    → 左のシークレットパネルでトグルがONになっているか確認してください')
    except Exception as _e_step1:
        print(f'  ❌ Step 1 失敗: {_e_step1}')

# Step 2: Google Driveの既知パスを探す
if not os.path.exists(_csj_path):
    print('  [Step 2] Google Driveで client_secrets.json を検索中...')
    _drive_candidates = [
        '/content/drive/MyDrive/client_secrets.json',
        '/content/drive/MyDrive/client_secrets_youtube.json',
        '/content/drive/MyDrive/credentials/client_secrets.json',
    ]
    for _dp in _drive_candidates:
        if os.path.exists(_dp):
            import shutil as _shutil_cs
            _shutil_cs.copy(_dp, _csj_path)
            print(f'  🔑 client_secrets.json をDriveからコピーしました: {_dp}')
            break
    else:
        print('  ⚠ Driveで client_secrets.json が見つかりませんでした')

# Step 3: ファイルアップロード（最終手段 — Steps 1・2 が失敗した場合のみ実行）
if not os.path.exists(_csj_path):
    print('  [Step 3] ファイルアップローダーを起動します...')
    print('  ⬆ 下に表示されるファイル選択ボタンから client_secrets.json を選択してください')
    try:
        from google.colab import files as _colab_files
        _uploaded = _colab_files.upload()
        if _uploaded:
            _fname = list(_uploaded.keys())[0]
            import json as _json_up
            _json_up.loads(_uploaded[_fname])  # JSON構文チェック
            with open(_csj_path, 'wb') as _f_up:
                _f_up.write(_uploaded[_fname])
            print(f'  🔑 client_secrets.json をアップロードしました（{_fname}）')
        else:
            print('  ⚠ ファイルが選択されませんでした')
    except Exception as _e_step3:
        print(f'  ❌ Step 3 失敗: {_e_step3}')

# 最終チェック: いずれかのステップで取得できていれば通過
if not os.path.exists(_csj_path):
    raise RuntimeError(
        '\n' + '='*55 + '\n'
        '❌  停止: client_secrets.json を取得できませんでした\n'
        '='*55 + '\n'
        '以下のいずれかを試してください:\n'
        '  A) 左のシークレットパネルで CLIENT_SECRETS_JSON を登録し\n'
        '     トグルをONにしてセル1を再実行\n'
        '  B) client_secrets.json を Google Drive の MyDrive に配置\n'
        '     してセル1を再実行\n'
        '  C) セル6（貼り付けセル）を使ってJSONを直接入力\n'
        + '='*55
    )
print('  ✅ client_secrets.json: OK')

# ── テーマ分類関数 ─────────────────────────────────────────
_YASERU_KW = ['ダイエット','食事','健康','ストレッチ','筋トレ','美容','腸活','スキンケア',
               '睡眠','栄養','カロリー','レシピ','体重','脂肪','運動','ヨガ','トレーニング',
               '肌','代謝','デトックス','断食','プロテイン','ビタミン','腸内','ウォーキング']
_OKANE_KW  = ['お金','投資','NISA','副業','節約','資産','株','貯金','収入','FX','不動産',
               '保険','税金','年金','家計','iDeCo','運用','利益','配当','ETF','積立','借金',
               'ローン','給料','転職','起業','フリーランス','稼ぐ','確定申告','クレジット']

def classify_theme_to_channel(theme):
    if FORCE_CHANNEL in ('yaseru','okane'):
        return FORCE_CHANNEL
    for kw in _YASERU_KW:
        if kw in theme: return 'yaseru'
    for kw in _OKANE_KW:
        if kw in theme: return 'okane'
    return 'yaseru'  # デフォルト

# テーマリスト表示
print(f'🔑 APIキー読み込み元: {_src}')
# ElevenLabs キー診断
_el_key_diag = globals().get('ELEVENLABS_API_KEY','').strip()
if _el_key_diag:
    _masked = _el_key_diag[:4] + '...' + _el_key_diag[-4:] if len(_el_key_diag)>=8 else '(短すぎ)'
    print(f'  🎙 ElevenLabs APIキー: {_masked} (長さ={len(_el_key_diag)}文字)')
else:
    print('  ⚠ ElevenLabs APIキー: 未設定 → gTTSフォールバック')
    print('    設定手順: 左のシークレットパネルで ELEVENLABS_API_KEY を登録してトグルをON')
# チャンネルID診断
_y_id = globals().get('YASERU_CHANNEL_ID','').strip()
_o_id = globals().get('OKANE_CHANNEL_ID','').strip()
if _y_id: print(f'  📺 やせる習慣図鑑 チャンネルID: {_y_id[:6]}...{_y_id[-4:]}')
else:      print('  ⚠ YASERU_CHANNEL_ID 未設定 → チャンネル名で自動判定')
if _o_id: print(f'  📺 お金の教科書 チャンネルID: {_o_id[:6]}...{_o_id[-4:]}')
else:      print('  ⚠ OKANE_CHANNEL_ID 未設定 → チャンネル名で自動判定')
print(f'  テーマ一覧:')
for _t in THEMES:
    _ch = classify_theme_to_channel(_t)
    _chn = 'やせる習慣図鑑' if _ch=='yaseru' else 'お金の教科書'
    print(f'    「{_t}」 → {_chn}')
print(f'  1テーマあたり: {VIDEOS_PER_THEME}本 / {DURATION}秒')
print(f'  自動投稿: {"ON" if AUTO_UPLOAD else "OFF"} / {UPLOAD_PRIVACY}')
print()
print('✅ 設定完了！')


In [ ]:
# 【自動】必要なツールをインストール（触らなくてOK）
!pip install -q gtts requests opencv-python-headless google-genai pillow google-api-python-client google-auth-oauthlib elevenlabs
!apt-get install -q -y ffmpeg fonts-noto-cjk p7zip-full
print('✅ ツールのインストール完了')

# ── VOICEVOX Engine セットアップ（ElevenLabsがない場合のみ）──
import subprocess as _sp, os as _os, time as _time
import requests as _req
VV_PORT = 50021
VV_URL  = f'http://127.0.0.1:{VV_PORT}'
VV_DIR  = '/content/voicevox_engine'
VV_SPEAKER = 3  # ずんだもん（ノーマル）
VV_AVAILABLE = False

# ElevenLabsが使える場合はVOICEVOXをスキップ（ダウンロード不要）
_el_key = globals().get('ELEVENLABS_API_KEY', '').strip()
if not _el_key:
    try:
        from google.colab import userdata as _ud_vv
        _el_key = (_ud_vv.get('ELEVENLABS_API_KEY') or '').strip()
    except Exception: pass

if _el_key:
    print('🎙 ElevenLabs APIキーあり → VOICEVOXをスキップ（ダウンロード不要）')
elif not _os.path.exists(f'{VV_DIR}/run'):
    print('🔊 VOICEVOXエンジンをダウンロード中（初回のみ・数分かかります）...')
    try:
        _rel = _req.get(
            'https://api.github.com/repos/VOICEVOX/voicevox_engine/releases/latest',
            timeout=30).json()
        # v0.20以降は .7z.001 形式（Linux x64 CPU版）
        _asset = next(
            (a for a in _rel['assets']
             if 'linux-cpu-x64' in a['name'] and a['name'].endswith('.7z.001')),
            None
        )
        if _asset is None:
            raise RuntimeError(f'Linux CPU x64アセットが見つかりません。利用可能: {[a["name"] for a in _rel["assets"]]}')
        _fn = f'/tmp/{_asset["name"]}'
        print(f'  ダウンロード: {_asset["name"]}')
        _sp.run(['wget', '-q', '--show-progress', '-O', _fn,
                 _asset['browser_download_url']], check=True)
        _os.makedirs(VV_DIR, exist_ok=True)
        _sp.run(['7z', 'x', _fn, f'-o{VV_DIR}', '-y'],
                check=True, capture_output=True)
        _os.remove(_fn)
        # 7z展開でサブディレクトリが作られた場合は1段上げる
        _subdirs = [d for d in _os.listdir(VV_DIR)
                    if _os.path.isdir(f'{VV_DIR}/{d}') and not d.startswith('.')]
        if _subdirs and not _os.path.exists(f'{VV_DIR}/run'):
            _sub = f'{VV_DIR}/{_subdirs[0]}'
            for _item in _os.listdir(_sub):
                _sp.run(['mv', f'{_sub}/{_item}', VV_DIR], check=True)
            _os.rmdir(_sub)
        _sp.run(['chmod', '+x', f'{VV_DIR}/run'], check=False)
        print('✅ VOICEVOXダウンロード・展開完了')
    except Exception as _e:
        print(f'⚠️ VOICEVOXダウンロード失敗（gTTSで代替します）: {_e}')

if not _el_key and _os.path.exists(f'{VV_DIR}/run'):
    _sp.Popen(
        [f'{VV_DIR}/run', '--host', '127.0.0.1', '--port', str(VV_PORT)],
        stdout=_sp.DEVNULL, stderr=_sp.DEVNULL
    )
    print('🔊 VOICEVOXエンジン起動中（最大60秒）...')
    for _i in range(60):
        _time.sleep(1)
        try:
            if _req.get(f'{VV_URL}/version', timeout=2).status_code == 200:
                VV_AVAILABLE = True
                print(f'✅ VOICEVOX起動完了（{_i+1}秒）→ 高品質音声モード')
                break
        except Exception:
            pass  # 接続待ち中の例外は正常（サーバー起動中）
    if not VV_AVAILABLE:
        print('⚠️ VOICEVOX起動タイムアウト → gTTSにフォールバック')
elif not _el_key:
    print('⚠️ VOICEVOXが見つかりません → gTTSで音声生成します')


In [ ]:
# 【切断防止】Colabの自動切断を防ぐ（動画生成中に実行しておく）
# ※ このセルを実行してからセル4（動画生成）を実行してください
import threading, time

_keepalive_running = True

def _keepalive_loop():
    """90秒ごとにColabのアイドル検知をリセット"""
    import IPython
    count = 0
    while _keepalive_running:
        time.sleep(90)
        count += 1
        IPython.display.display(IPython.display.Javascript(
            'google.colab.kernel.proxyPort(0)'
        ))
        print(f'\r⏳ 切断防止 keepalive #{count} ({count*90//60}分経過)', end='', flush=True)

_t = threading.Thread(target=_keepalive_loop, daemon=True)
_t.start()
print('✅ 切断防止スタート（動画生成が終わったら気にしなくてOK）')


In [ ]:
# 【自動】① キーワード調査 → ② Shortsトレンド → ③ 長尺・最新情報 → ④ AI統合分析 → ⑤ 切り口生成

import requests as _req, json as _json, re, datetime

# ── 情報取得関数 ─────────────────────────────────────────────

def fetch_search_keywords(theme):
    """ユーチューブオートコンプリートで「今まさに検索されているワード」を取得（APIキー不要）"""
    try:
        r = _req.get(
            'https://suggestqueries.google.com/complete/search',
            params={'client': 'youtube', 'hl': 'ja', 'ds': 'yt', 'q': theme},
            timeout=10
        )
        if r.status_code != 200:
            return []
        text = r.text
        if text.startswith('window.'):
            text = text[text.index('(') + 1:text.rindex(')')]
        data = _json.loads(text)
        return [item[0] for item in data[1] if isinstance(item, (list, tuple)) and item][:12]
    except Exception as e:
        print(f'  ⚠ キーワード取得失敗: {e}')
        return []

def fetch_youtube_shorts_trends(theme):
    """2パス検索: ①直近90日トレンド(relevance) + ②全期間人気(viewCount) → 重複排除"""
    after = (datetime.datetime.utcnow() - datetime.timedelta(days=90)).strftime('%Y-%m-%dT%H:%M:%SZ')
    base = {
        'part': 'snippet', 'q': f'{theme} shorts', 'type': 'video',
        'regionCode': 'JP', 'relevanceLanguage': 'ja',
        'key': YOUTUBE_API_KEY, 'videoDuration': 'short',
    }
    passes = [
        {**base, 'order': 'relevance', 'maxResults': 15, 'publishedAfter': after},
        {**base, 'order': 'viewCount',  'maxResults': 10},
    ]
    results, seen = [], set()
    for params in passes:
        try:
            r = _req.get('https://www.googleapis.com/youtube/v3/search', params=params, timeout=15)
            if r.status_code != 200:
                print(f'  ⚠ Shorts検索API: {r.status_code}')
                continue
            for item in r.json().get('items', []):
                vid = item['id'].get('videoId', '')
                if vid and vid not in seen:
                    seen.add(vid)
                    results.append({
                        'id':      vid,
                        'title':   item['snippet'].get('title', ''),
                        'channel': item['snippet'].get('channelTitle', ''),
                    })
        except Exception as e:
            print(f'  ⚠ YouTube API失敗: {e}')
    return results

def fetch_longform_content(theme):
    """長尺動画（4～20分）から最新エビデンス・新成分・トレンド知識を収集"""
    after = (datetime.datetime.utcnow() - datetime.timedelta(days=180)).strftime('%Y-%m-%dT%H:%M:%SZ')
    queries = [
        f'{theme} 最新 効果 エビデンス',
        f'{theme} 方法 やり方 解説',
    ]
    results, seen = [], set()
    for q in queries:
        try:
            r = _req.get(
                'https://www.googleapis.com/youtube/v3/search',
                params={
                    'part': 'snippet', 'q': q, 'type': 'video',
                    'order': 'relevance', 'regionCode': 'JP',
                    'relevanceLanguage': 'ja', 'maxResults': 8,
                    'key': YOUTUBE_API_KEY, 'videoDuration': 'medium',
                    'publishedAfter': after,
                },
                timeout=15
            )
            if r.status_code == 200:
                for item in r.json().get('items', []):
                    vid = item['id'].get('videoId', '')
                    title = item['snippet'].get('title', '')
                    if vid and vid not in seen and title:
                        seen.add(vid)
                        results.append(title)
        except Exception as e:
            print(f'  ⚠ 長尺動画取得失敗: {e}')
    return results[:15]

def fetch_video_stats(video_ids):
    """再生数・いいね・コメ・タグ・尺を取得。60秒超えはShortsではないので除外。"""
    if not video_ids: return []
    def _parse_dur(iso):
        m = re.search(r'PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?', iso or '')
        if not m: return 999
        h, mn, s = (int(x or 0) for x in m.groups())
        return h * 3600 + mn * 60 + s
    try:
        r = _req.get(
            'https://www.googleapis.com/youtube/v3/videos',
            params={'part': 'snippet,statistics,contentDetails',
                    'id': ','.join(video_ids[:20]), 'key': YOUTUBE_API_KEY},
            timeout=15
        )
        if r.status_code != 200:
            print(f'  ⚠ YouTube統計API: {r.status_code}')
            return []
        results = []
        for item in r.json().get('items', []):
            snip  = item.get('snippet', {})
            stats = item.get('statistics', {})
            dur   = _parse_dur(item.get('contentDetails', {}).get('duration', ''))
            if dur > 60: continue
            results.append({
                'title':       snip.get('title', ''),
                'description': snip.get('description', '')[:200],
                'tags':        snip.get('tags', [])[:8],
                'duration_s':  dur,
                'views':       int(stats.get('viewCount',   0)),
                'likes':       int(stats.get('likeCount',   0)),
                'comments':    int(stats.get('commentCount',0)),
            })
        return sorted(results, key=lambda x: x['views'], reverse=True)
    except Exception as e:
        print(f'  ⚠ 統計取得失敗: {e}')
        return []

CLAUDE_URL = 'https://api.anthropic.com/v1/messages'

def call_ai(prompt, tokens=4096):
    res = _req.post(
        CLAUDE_URL,
        headers={'x-api-key': CLAUDE_API_KEY,
                 'anthropic-version': '2023-06-01',
                 'content-type': 'application/json'},
        json={'model': 'claude-haiku-4-5-20251001',
              'max_tokens': tokens,
              'messages': [{'role': 'user', 'content': prompt}]},
        timeout=120
    )
    if res.status_code != 200:
        err = res.text[:300].replace(CLAUDE_API_KEY, '***')
        raise RuntimeError(f'Claude APIエラー ({res.status_code}): {err}')
    _data = res.json()
    if not _data.get('content'):
        raise RuntimeError(f'Claude API応答が空: {str(_data)[:200]}')
    return _data['content'][0]['text']

def clean_title(t):
    t = re.sub(r'^#+\s*', '', t)
    t = re.sub(r'\*+', '', t)
    t = re.sub(r'^\d+[\.)\]]\s*', '', t)
    return t.strip()


def generate_angles_for_theme(theme):
    """1テーマ分のリサーチ＋ANGLES生成を実行して返す"""
    global VIDEO_COUNT, ANGLES, DIAGNOSIS_LINES
    THEME = theme  # ローカルで上書き
    
    # ══════════════════════════════════════════════════════
    # ① 検索キーワード調査（オートコンプリート）
    # ══════════════════════════════════════════════════════
    print(f'🔍 「{THEME}」の検索キーワードを調査中...')
    kw_suggestions = fetch_search_keywords(THEME)
    kw_text = '・'.join(kw_suggestions) if kw_suggestions else 'データなし'
    print(f'  ✓ 検索ワード({len(kw_suggestions)}件): {kw_text[:80]}')
    
    # ══════════════════════════════════════════════════════
    # ② YouTube Shorts トレンド取得（直近90日 + 歴代人気）
    # ══════════════════════════════════════════════════════
    print('\n📺 YouTube Shortsトレンドを検索中...')
    yt_basic = fetch_youtube_shorts_trends(THEME)
    
    if yt_basic:
        print(f'  ✓ {len(yt_basic)}件の動画を発見')
        yt_stats = fetch_video_stats([v['id'] for v in yt_basic])
        if yt_stats:
            print(f'  ✓ Shorts確認済み {len(yt_stats)}本の詳細取得完了')
            print('\n  🔥 最も再生されている動画:')
            for v in yt_stats[:3]:
                print(f'    再生:{v["views"]:,} │ {v["title"][:45]}')
            analysis_lines = []
            for i, v in enumerate(yt_stats[:15]):
                tags_str = '・'.join(v['tags'][:6]) if v['tags'] else 'タグなし'
                analysis_lines.append(
                    f'{i+1}. 「{v["title"]}」{v.get("duration_s","")}秒\n'
                    f'   再生:{v["views"]:,} / いいね:{v["likes"]:,} / コメ:{v["comments"]:,}\n'
                    f'   タグ: {tags_str}\n'
                    f'   説明: {v["description"][:100]}'
                )
            yt_data_text = '\n'.join(analysis_lines)
        else:
            yt_data_text = '\n'.join(f'{i+1}. 「{v["title"]}」' for i, v in enumerate(yt_basic))
    else:
        print('  ⚠ Shorts取得できず。AI知識でトレンド分析します。')
        yt_data_text = f'テーマ「{THEME}」の一般的なトレンド情報'
    
    # ══════════════════════════════════════════════════════
    # ③ 長尺動画から最新エビデンス・トレンド情報を収集
    # ══════════════════════════════════════════════════════
    print('\n📹 長尺動画（最新解説・エビデンス・製品情報）を調査中...')
    longform_titles = fetch_longform_content(THEME)
    if longform_titles:
        print(f'  ✓ {len(longform_titles)}件の長尺動画を発見')
        longform_text = '\n'.join(f'・{t}' for t in longform_titles)
    else:
        print('  ⚠ 長尺動画データなし')
        longform_text = 'データなし'
    
    # ══════════════════════════════════════════════════════
    # ④ Claude AI統合分析（Shorts + 長尺 + キーワード 三点合わせ）
    # ══════════════════════════════════════════════════════
    print('\n🤖 Claude AIで統合分析中...')
    trend = call_ai(
        f'テーマ「{THEME}」のYouTube総合リサーチデータ\n\n'
        f'【A. 今まさに検索されているキーワード（オートコンプリート実測値）】\n{kw_text}\n\n'
        f'【B. Shorts上位動画（直近90日トレンド＋歴代人気 混合・実数値）】\n{yt_data_text}\n\n'
        f'【C. 長尺動画タイトル（最新解説・エビデンス・製品・成分情報源）】\n{longform_text}\n\n'
        f'上要3つのデータを統合して、以下を日本語・箇条書きで出力してください:\n\n'
        f'　1. 今バズるタイトル公式（3～5パターン）、\n'
        f'実際のデータから抽出した構造（数字＋煽り語の組み合わせ、具体例付き）\n\n'
        f'　2. 冠頭3秒フックフレーズ（5例）\n'
        f'視聴者が手を止める具体的な日本語フレーズ\n\n'
        f'　3. 今ホットな最新ネタ（Shortsに落とし込める情報）\n'
        f'長尺動画から抽出した最新エビデンス・流行成分・注目メソッドをShorts1シーン分に要約\n\n'
        f'　4. コメント誘発ネタ\n'
        f'「それ違う」「私もやってみる」が生まれやすいテーマ・断言・比較\n\n'
        f'　5. タイトルに入れるべき検索ワード（自然に入るもの）\n'
        f'オートコンプリートから抽出した、タイトルに入れると検索ヒットするワード\n\n'
        f'　6. 音声スタイル】語尾・テンポ・間の特徴（2行）'
    )
    print('  ✓ 統合分析完了')
    print('\n  ━━━━ 分析サマリー ━━━━')
    print(f'  {trend[:400]}')
    print(f'  ...')
    
    TREND_ANALYSIS = trend
    
    # ══════════════════════════════════════════════════════
    # ⑤ VIDEO_COUNT個の切り口を生成（キーワード自然組み込み）
    # ══════════════════════════════════════════════════════
    print(f'\n💡 切り口を{VIDEO_COUNT}個考案中...')
    angles_raw = call_ai(
        f'テーマ「{THEME}」のYouTube Shortsタイトルを{VIDEO_COUNT}個出力してください。\n\n'
        f'トレンド分析（実データ）:\n{trend[:700]}\n\n'
        f'実際に検索されているワード: {kw_text[:200]}\n\n'
        f'【絶対守ること】\n'
        f'・タイトルのみを出力（説明文・番号・記号なし）\n'
        f'・1行1タイトル\n'
        f'・20文字以内\n'
        f'・全て異なるフォーマットにする（同じ型を繰り返さない）\n'
        f'・「タイトルに入れるべき検索ワード」を自然に含める（無理に入れない）\n'
        f'・長尺動画から得た最新ネタ（エビデンス・新成分・メソッド）を優先\n\n'
        f'【使えるフォーマット例（混ぜて使うこと）】\n'
        f'①疑問形: 「〇〇は本当に効くの？」\n'
        f'②数字+具体: 「朝7時に〇〇するだけ」「3食これだけ食べた結果」\n'
        f'③比較・逆説: 「〇〇より△△の方が効果的な理由」「やめたら変わった」\n'
        f'④警告・発見: 「これ続けると逆効果」「医師が驚いた〇〇」\n'
        f'⑤ハウツー: 「〇〇を食べる順番」「寝る前5分でできる〇〇」\n\n'
        f'【禁止パターン】\n'
        f'・「〜キロやってた人」「〜キロ落とした人」「〜キロ減った人」系\n'
        f'・「知らないと損」の多用（1本に1回まで）\n'
        f'・同じフォーマットを複数本で繰り返す\n'
        f'・「○選」「○つ」をタイトルに使う場合は動画内で全て完結させること\n'
        f'・金融系タイトルにビットコイン・仮想通貨・FXは使わない\n\n'
        f'出力例（ダイエットの場合）:\n'
        f'プロテイン飲む本当のタイミング\n'
        f'食べる順番を変えたら体重が落ちた\n'
        f'寝る前にこれをやめるだけ\n'
        f'腸活で痩せた人の共通点とは\n'
        f'朝食抜きが逆効果な理由',
        tokens=1024
    )
    ANGLES = [clean_title(l) for l in angles_raw.strip().split('\n')
              if l.strip() and len(l.strip()) > 3][:VIDEO_COUNT]
    while len(ANGLES) < VIDEO_COUNT:
        ANGLES.append(f'{THEME}の秘密 Vol.{len(ANGLES)+1}')
    
    print(f'\n生成する{VIDEO_COUNT}本:')
    for i, a in enumerate(ANGLES):
        print(f'  {i+1:2d}. {a}')
    print(f'\n✅ 切り口の決定完了')
    
    
    # ── 診断系×ASMR フォーマット（VIDEO_FORMAT=='diagnosis' のとき）──
    if globals().get('VIDEO_FORMAT') == 'diagnosis':
        _ch_nm  = globals().get('CHANNEL_NAME', THEME)
        _n_item = globals().get('DIAGNOSIS_ITEMS', 4)
        _diag_p = (
            f'YouTube Shorts「診断系×ASMR」動画の台本を生成してください。\n'
            f'チャンネル: {_ch_nm}\n'
            f'テーマ: {THEME}\n'
            f'診断項目数: {_n_item}\n\n'
            f'【厳守する構成 (全{2+_n_item*2+2}行)】\n'
            f'1行目: 診断タイトル（例: おデブ習慣診断）\n'
            f'2行目: 煽りコピー（例: 3つ以上は一生痩せません）\n'
            f'3行目〜{2+_n_item*2}行目: 診断項目×{_n_item}（①行動[10字以内]+1行解説[8字以内] を繰り返す）\n'
            f'{3+_n_item*2}行目: 当てはまった数は？\n'
            f'{4+_n_item*2}行目: コメントで教えてね！\n\n'
            f'【絶対ルール】1行10文字以内 / 絵文字NG / 台本の行のみ出力（説明不要）'
        )
        _diag_raw = call_ai(_diag_p, tokens=400)
        DIAGNOSIS_LINES = [l.strip() for l in _diag_raw.strip().split('\n')
                           if l.strip() and 1 <= len(l.strip()) <= 15][:4 + _n_item * 2]
        # diagnosis フォーマット用にANGLESを上書き
        ANGLES = [f'{THEME}診断_v{i+1}' for i in range(VIDEO_COUNT)]
        print(f'\n📋 診断系台本（{len(DIAGNOSIS_LINES)}行）:')
        for _di, _dl in enumerate(DIAGNOSIS_LINES):
            print(f'  {_di+1:2d}. {_dl}')
        print(f'\n✅ 診断系フォーマット準備完了（{VIDEO_COUNT}本）')
    else:
        DIAGNOSIS_LINES = []
    return ANGLES

# デフォルト実行（Cell1のTHEMEで動かす）
generate_angles_for_theme(globals().get('THEME', THEMES[0] if THEMES else 'ダイエット'))


In [ ]:
# 【自動】③〜⑥ 全動画を一括生成（触らなくてOK）

import cv2, numpy as np, subprocess, tempfile, time, wave, json as _json, shutil as _shutil
import scipy.io.wavfile as wio, requests as _req, re
from pathlib import Path
from gtts import gTTS

def anime(img):
    c = cv2.bilateralFilter(cv2.bilateralFilter(img,9,250,250),9,250,250)
    g = cv2.medianBlur(cv2.cvtColor(img,cv2.COLOR_BGR2GRAY),5)
    e = cv2.cvtColor(cv2.adaptiveThreshold(g,255,cv2.ADAPTIVE_THRESH_MEAN_C,cv2.THRESH_BINARY,9,5),cv2.COLOR_GRAY2BGR)
    r = cv2.bitwise_and(c,e)
    h = cv2.cvtColor(r,cv2.COLOR_BGR2HSV).astype(np.float32)
    h[:,:,1] = np.clip(h[:,:,1]*1.5,0,255)
    return cv2.cvtColor(h.astype(np.uint8),cv2.COLOR_HSV2BGR)

def manga(img):
    g = cv2.createCLAHE(2.0,(8,8)).apply(cv2.cvtColor(img,cv2.COLOR_BGR2GRAY))
    e = cv2.dilate(cv2.Canny(cv2.GaussianBlur(g,(3,3),0),30,100),np.ones((2,2),np.uint8))
    _,t = cv2.threshold(g,180,255,cv2.THRESH_BINARY)
    return cv2.cvtColor(cv2.addWeighted(t,.75,cv2.bitwise_not(e),.25,0),cv2.COLOR_GRAY2BGR)

def illust(img):
    c = cv2.bilateralFilter(img,15,80,80)
    h = cv2.cvtColor(c,cv2.COLOR_BGR2HSV).astype(np.float32)
    h[:,:,1]=np.clip(h[:,:,1]*1.7,0,255); h[:,:,2]=np.clip(h[:,:,2]*1.1,0,255)
    c = cv2.cvtColor(h.astype(np.uint8),cv2.COLOR_HSV2BGR)
    e = cv2.cvtColor(cv2.adaptiveThreshold(cv2.medianBlur(cv2.cvtColor(img,cv2.COLOR_BGR2GRAY),7),255,
        cv2.ADAPTIVE_THRESH_MEAN_C,cv2.THRESH_BINARY,11,9),cv2.COLOR_GRAY2BGR)
    return cv2.bitwise_and(c,e)

STYLES = {'anime':anime,'manga':manga,'illustration':illust}

def ff(*a):
    r = subprocess.run(['ffmpeg','-y',*[str(x) for x in a]],
                       capture_output=True,text=True,timeout=600)
    if r.returncode != 0:
        raise RuntimeError(r.stderr[-1200:])

def at(s):
    return f'{int(s//3600)}:{int((s%3600)//60):02d}:{int(s%60):02d}.{int((s%1)*100):02d}'

def get_wav_dur(path):
    # wave.open はWAVのみ対応。MP3/M4Aはffprobeで取得
    try:
        with wave.open(str(path),'r') as wf:
            return wf.getnframes()/wf.getframerate()
    except Exception:
        pass
    try:
        r = subprocess.run(
            ['ffprobe','-v','quiet','-show_entries','format=duration',
             '-of','default=noprint_wrappers=1:nokey=1',str(path)],
            capture_output=True, text=True, timeout=15)
        d = float(r.stdout.strip())
        if d > 0:
            return d
    except Exception:
        pass
    return 3.0

def clean_text(t):
    t = re.sub(r'#\S+', '', t)
    t = re.sub(r'\[速く\]|\[ゆっくり\]|\[強調\]','',t)
    t = re.sub(r'\[間\d+\.?\d*\]','、',t)
    t = re.sub(r'（ここにセリフ）|\(ここにセリフ\)|シーン\d+[（(][^)）]*[)）]\s*[:：]','',t)
    t = re.sub(r'^#+\s*','',t); t = re.sub(r'[\*_]','',t)
    return re.sub(r'\s+',' ',t).strip()

def verify_and_fix_script(scene_texts, theme, angle):
    """Step3: 文法・内容・フォーマットを自己検証 → 問題箇所を自動修正"""
    if not scene_texts:
        return scene_texts
    scenes_str = '\n'.join(f'シーン{i+1}: {t}' for i, t in enumerate(scene_texts))
    try:
        result = call_ai(
            f'以下のYouTube Shorts台本を検証し、修正したものを出力してください。\n\n'
            f'【台本】\n{scenes_str}\n\n'
            f'【テーマ】{theme}  【タイトル】{angle}\n\n'
            f'【検証・修正ルール（この順番で全シーン確認）】\n'
            f'①語尾が「～んだ」「～なんです」「～でしょう」「～むんだよ」など\n'
            f'  ロボット調なら「～だよ！」「～してみて！」「～じゃない？」に修正\n'
            f'②英語・記号・ハッシュタグが混入していたら削除\n'
            f'③15文字を超えるセリフは意味を保ったまま短縮\n'
            f'④シーン1のフックが弱ければ「え、マジ？」「知ってた？」型に強化\n'
            f'⑤内容の流れ（駅き→共感→解決→行動）が不自然なら並び替え\n\n'
            f'【出力フォーマット（厳守）】\n'
            f'修正後のセリフのみ出力。問題がないシーンもそのまま全て出力。\n'
            f'シーン1: （修正後セリフ）\n'
            f'シーン2: （修正後セリフ）\n'
            f'（元と同じシーン数で出力すること）',
            tokens=512
        )
        fixed = []
        for line in result.strip().split('\n'):
            m = re.match(r'シーン\d+[::：]\s*(.+)', line.strip())
            if m:
                t = clean_text(m.group(1).strip())
                if t: fixed.append(t)
        if len(fixed) == len(scene_texts):
            changed = sum(1 for a, b in zip(scene_texts, fixed) if a != b)
            if changed:
                print(f'  ✏️  {changed}シーンを自動修正')
            return fixed
        return scene_texts
    except Exception as e:
        print(f'  ⚠ 検証スキップ: {e}')
        return scene_texts

def tts_text_cleanse(text):
    """TTS専用テキスト補正: gTTSが自然なイントネーションで読めるよう意味・語尾を修正"""
    # ① AI特有の語尾をgTTSが自然に読める語尾に変換
    text = re.sub(r'んだよ([！！]?)', r'よ\1', text)       # 〜んだよ！ → 〜よ！
    text = re.sub(r'んだよね', 'よね', text)
    text = re.sub(r'んだね([！！]?)', r'だよね\1', text)
    text = re.sub(r'んです([よ！ね。]?)', r'よ\1', text)    # 〜んです → 〜よ
    text = re.sub(r'なんだ([！！。]?)', r'だよ\1', text)   # 〜なんだ！ → 〜だよ！
    text = re.sub(r'なんです', 'だよ', text)
    text = re.sub(r'するんだ([！！。]?)', r'するよ\1', text)
    text = re.sub(r'できるんだ([！！。]?)', r'できるよ\1', text)
    text = re.sub(r'整うんだ([！！。]?)', r'整うよ\1', text)   # ユーザー指定例
    text = re.sub(r'変わるんだ([！！。]?)', r'変わるよ\1', text)  # ユーザー指定例
    text = re.sub(r'あるんだ([！！。]?)', r'あるよ\1', text)
    text = re.sub(r'いるんだ([！！。]?)', r'いるよ\1', text)
    # ② 読点（ポーズ）挿入: gTTSが自然な間を置けるよう感嘆詞の後に読点
    text = re.sub(r'^(え)(マジ|本当|すごい|やば)', r'\1、\2', text)  # えマジ→え、マジ
    text = re.sub(r'^(実は)([^、])', r'\1、\2', text)               # 実は→実は、
    text = re.sub(r'^(ねえ|ねっ|あのね)([^、])', r'\1、\2', text)   # ねえ→ねえ、
    text = re.sub(r'^(知ってる)(\?|？)?([^、])', r'\1？\3', text)
    # ③ 数字＋単位の間にスペース: gTTSが分けて読めるよう
    text = re.sub(
        r'([一二三四五六七八九十百千万億])(キロ|グラム|センチ|メートル|ヶ月|ヵ月|カ月|週間|日間|時間)',
        r'\1 \2', text
    )
    # ④ 重複記号を整理
    text = re.sub(r'！{2,}', '！', text)
    text = re.sub(r'、{2,}', '、', text)
    return text.strip()

def preprocess_tts(text):
    """ハッシュタグ・記号を完全除去し gTTS が読めるテキストへ"""
    text = re.sub(r'#\S+', '', text)
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'[\[\](){}@&*|\\^~`<>=+_・]', '', text)
    text = re.sub(r'[^ -~　-鿿！-｠゠-ヿ぀-ゟ]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    if len(text) > 16 and '、' not in text:
        mid = len(text) // 2
        for j in range(max(0, mid-4), min(len(text)-1, mid+5)):
            if text[j] in 'はがをにでもよねてし':
                text = text[:j+1] + '、' + text[j+1:]
                break
    return text.strip() or 'つぎのシーンです'

def preprocess_for_elevenlabs(text):
    """ElevenLabs専用: 漢字の誤英語読み・造語防止のための変換"""
    import re as _re
    # ElevenLabsが英語読みしやすい漢字をひらがなに変換
    _kanji_map = [
        ('体力', 'たいりょく'), ('体内', 'たいない'), ('体脂肪', 'たいしぼう'),
        ('タンパク質', 'たんぱくしつ'), ('タンパク', 'たんぱく'),
        ('代謝', 'たいしゃ'), ('脂肪', 'しぼう'), ('筋肉', 'きんにく'),
        ('睡眠', 'すいみん'), ('疲労', 'ひろう'), ('摂取', 'せっしゅ'),
        ('分泌', 'ぶんぴつ'), ('血糖値', 'けっとうち'), ('インスリン', 'いんすりん'),
        ('コルチゾール', 'こるちぞーる'), ('ホルモン', 'ほるもん'),
        ('資産', 'しさん'), ('投資', 'とうし'), ('利回り', 'りまわり'),
        ('複利', 'ふくり'), ('節税', 'せつぜい'), ('確定申告', 'かくていしんこく'),
    ]
    for _k, _h in _kanji_map:
        text = text.replace(_k, _h)
    # 数字+時 → 読みやすく（11時→じゅういちじ等は不要、数字はそのままでOK）
    # アラビア数字の前後に半角スペースで読み区切りを強制
    text = _re.sub(r'(\d+)(時間?|分|秒|日|週|ヶ月|カ月|年|キロ|グラム|リットル|kcal|kg)',
                   r'\1 \2', text)
    return text

def make_tts_voicevox(text, tmp_dir, prefix, speaker=None):
    """VOICEVOX REST APIで高品質日本語音声を生成（ずんだもんノーマル）"""
    if speaker is None: speaker = VV_SPEAKER
    try:
        import requests as _r
        # Step1: audio_query
        q = _r.post(f'{VV_URL}/audio_query',
                    params={'text': text, 'speaker': speaker}, timeout=30)
        q.raise_for_status()
        query = q.json()
        # 話速・イントネーション・無音長を調整
        _vcfg = globals().get('_ACTIVE_VOICE_CFG', _VOICE_DEFAULT)
        query['speedScale']       = _vcfg['speed_scale']
        query['intonationScale']  = _vcfg['intonation']
        query['prePhonemeLength'] = 0.05
        query['postPhonemeLength']= 0.10
        # Step2: synthesis
        s = _r.post(f'{VV_URL}/synthesis',
                    params={'speaker': speaker}, json=query, timeout=60)
        s.raise_for_status()
        out = tmp_dir / f'{prefix}_vv.wav'
        out.write_bytes(s.content)
        return out
    except Exception as _e:
        print(f'  ⚠ VOICEVOX TTS失敗: {_e}')
        return None


# ── ElevenLabs TTS（高品質音声）────────────────────────────────────
def make_tts_elevenlabs(text, tmp_dir, prefix):
    """ElevenLabs APIで音声生成（ELEVENLABS_API_KEY設定時に使用）"""
    _key = globals().get('ELEVENLABS_API_KEY', '').strip()
    if not _key:
        return None
    try:
        import requests as _rel
        # 日本語対応: eleven_multilingual_v2 + Bella (多言語対応ボイス)
        _voice_id = globals().get('ELEVENLABS_VOICE_ID', 'EXAVITQu4vr4xnSDxMaL')  # Sarah（多言語）
        _r = _rel.post(
            f'https://api.elevenlabs.io/v1/text-to-speech/{_voice_id}',
            headers={'xi-api-key': _key, 'Content-Type': 'application/json'},
            json={'text': text, 'model_id': 'eleven_multilingual_v2',
                  'voice_settings': globals().get('_EL_VOICE_SETTINGS', {
                      'stability': 0.45, 'similarity_boost': 0.75,
                      'style': 0.50, 'use_speaker_boost': True})},
            timeout=60
        )
        if _r.status_code == 200:
            _mp3 = tmp_dir / f'{prefix}_el.mp3'
            _mp3.write_bytes(_r.content)
            return _mp3
        else:
            _masked_k = (_key[:4]+'...'+_key[-4:]) if len(_key)>=8 else '(短)'
            print(f'  ⚠ ElevenLabs失敗 status={_r.status_code} key={_masked_k}: {_r.text[:200]}')
            if _r.status_code == 429:
                print('    → APIの使用制限に達しました。しばらく待ってから再実行してください。')
            elif _r.status_code in (401, 403):
                print(f'    → APIキーが無効(401)。ElevenLabsダッシュボードで新しいキーを作成し')
                print(f'    → Colabシークレット ELEVENLABS_API_KEY に貼り付けてトグルをONに。')
                print(f'    → 現在のキー長: {len(_key)}文字 / 先頭: {_key[:6] if len(_key)>=6 else _key}')
            elif _r.status_code == 422:
                print('    → voice_idが無効です。ELEVENLABS_VOICE_IDを確認してください。')
            return None
    except Exception as _e:
        print(f'  ⚠ ElevenLabs例外: {_e}')
        return None

def make_tts_audio(text, tmp_dir, prefix):
    """ElevenLabs優先 → VOICEVOX → gTTSフォールバックで音声生成"""
    text = preprocess_tts(text)
    # ── ElevenLabs優先ルート ──────────────────────────────────────────
    if globals().get('ELEVENLABS_API_KEY', '').strip():
        _el = make_tts_elevenlabs(text, tmp_dir, prefix)
        if _el and _el.exists():
            return _el
    # ── VOICEVOX優先ルート ───────────────────────────────────
    if VV_AVAILABLE:
        _spk = globals().get('_ACTIVE_VOICE_CFG', _VOICE_DEFAULT)['vv_speaker']
        _vv = make_tts_voicevox(text, tmp_dir, prefix, speaker=_spk)
        if _vv and _vv.exists():
            return _vv
    # ── gTTSフォールバック ────────────────────────────────────
    sr = 44100
    # 句読点の後で分割（句読点を各フレーズに含める）
    parts = [p.strip() for p in re.split(r'(?<=[。！？、])', text) if p.strip() and len(p.strip()) >= 2]
    if not parts:
        parts = [text] if text else []
    if not parts:
        return None

    wavs = []
    for j, phrase in enumerate(parts):
        mp3 = tmp_dir/f'{prefix}_{j}.mp3'
        wav = tmp_dir/f'{prefix}_{j}.wav'
        sil = tmp_dir/f'{prefix}_{j}_s.wav'
        try:
            gTTS(text=phrase, lang='ja', slow=False).save(str(mp3))
            ff('-i',str(mp3),'-filter:a',
               'silenceremove=start_periods=1:start_silence=0.02:start_threshold=-42dB'
               f':stop_periods=-1:stop_silence=0.04:stop_threshold=-42dB,'
               f'atempo={globals().get("_ACTIVE_VOICE_CFG", _VOICE_DEFAULT)["atempo"]}',
               '-ar','44100','-ac','1',str(wav))
            wavs.append(wav)
            # 句点・感嘆符・疑問符 = 180ms、読点 = 80ms の無音
            sil_ms = 140 if phrase[-1] in '。！？' else 60
            wio.write(str(sil), sr, np.zeros(int(sr*sil_ms/1000), dtype=np.float32))
            wavs.append(sil)
        except Exception as _gtts_e:
            print(f'    ⚠ gTTSフレーズ生成失敗 (スキップ): {_gtts_e}')

    if not wavs:
        return None

    out = tmp_dir/f'{prefix}_out.wav'
    if len(wavs) == 1:
        _shutil.copy(str(wavs[0]), str(out))
    else:
        lf = tmp_dir/f'{prefix}_lf.txt'
        lf.write_text('\n'.join(f"file '{p}'" for p in wavs))
        ff('-f','concat','-safe','0','-i',str(lf),'-c','copy',str(out))
    return out

def elevenlabs_full_script(texts, tmp_dir):
    """全シーンテキストを1回のElevenLabsコールで生成しシーン数等分WAVリストを返す。
    英語読みを防ぎ数字・固有名詞のイントネーションを自然にする。"""
    _key = globals().get('ELEVENLABS_API_KEY', '').strip()
    if not _key or not texts:
        return None
    _cleaned = [preprocess_for_elevenlabs(preprocess_tts(tts_text_cleanse(t))) for t in texts]
    _full = '。'.join(t for t in _cleaned if t)
    if not _full:
        return None
    # ── チャンネル・感情強度に合わせてvoice_settingsを動的設定 ────────
    _ch = globals().get('_CURRENT_CHANNEL_LABEL', 'yaseru')
    _excl = sum(t.count('！') + t.count('!') for t in _cleaned)
    _ene  = sum(1 for t in _cleaned
               for w in ['やば','すごい','まじ','最強','革命','衝撃','絶対','驚','爆速','激変'] if w in t)
    _base = 0.65 if _ch == 'yaseru' else 0.32
    _style = min(0.92, _base + min(0.25, (_excl + _ene * 2) * 0.025))
    _stab  = round(max(0.20, 0.75 - _style * 0.55), 2)
    globals()['_EL_VOICE_SETTINGS'] = {
        'stability': _stab,
        'similarity_boost': 0.78,
        'style': round(_style, 2),
        'use_speaker_boost': True,
    }
    print(f'  🎭 感情設定: ch={_ch} / style={_style:.2f} / stability={_stab:.2f}')
    print(f'  🎤 ElevenLabs全文一括生成中（{len(texts)}シーン / {len(_full)}文字）...')
    _full_mp3 = make_tts_elevenlabs(_full, tmp_dir, 'el_full')
    if not _full_mp3 or not _full_mp3.exists():
        return None
    # MP3→WAV変換（get_wav_dur/whisper/atrimが正しく動くために必須）
    _full_wav = tmp_dir / 'el_full.wav'
    try:
        ff('-i', str(_full_mp3), '-ar', '44100', '-ac', '1', str(_full_wav))
    except Exception as _conv_e:
        print(f'  ⚠ MP3→WAV変換失敗: {_conv_e}')
        return None
    if not _full_wav.exists():
        return None
    _total = get_wav_dur(_full_wav)
    if _total < 1.0:
        return None
    n = len(texts)

    # ── Whisperでセグメント境界を検出 ──────────────────────────────────
    _boundaries = None
    try:
        import importlib, subprocess as _wsp
        if importlib.util.find_spec('whisper') is None:
            print('  📦 whisper インストール中...')
            _wsp.run(['pip','install','-q','openai-whisper'], check=True)
        import whisper as _ws, difflib
        _wm = _ws.load_model('tiny')
        _result = _wm.transcribe(str(_full_wav), language='ja', fp16=False)
        _transcribed = _result['text'].replace(' ', '')
        _original = ''.join(_cleaned)
        _sim = difflib.SequenceMatcher(None, _transcribed, _original).ratio()
        _sim_pct = int(_sim * 100)
        if _sim_pct < 35:
            print(f'  ❌ 日本語検証NG: 類似度{_sim_pct}% → シーン別生成にフォールバック')
            return None
        print(f'  ✅ 日本語検証: 類似度{_sim_pct}%')
        # Whisperセグメント終端を境界候補に使う
        _segs = _result.get('segments', [])
        _seg_ends = sorted(set(s['end'] for s in _segs if s['end'] < _total))
        if len(_segs) >= n:
            # テキスト長比率でターゲット境界を計算し、最近傍のセグメント終端にスナップ
            _lengths = [max(len(t), 1) for t in _cleaned]
            _total_len = sum(_lengths)
            _bds = [0.0]
            _cumul = 0
            for _l in _lengths[:-1]:
                _cumul += _l
                _target = _total * _cumul / _total_len
                # 最近傍のWhisperセグメント終端を選ぶ（±1.5s以内）
                _near = [e for e in _seg_ends if abs(e - _target) < 1.5]
                _bds.append(min(_near, key=lambda e: abs(e - _target)) if _near else _target)
            _bds.append(_total)
            _boundaries = _bds
            print(f'  ✅ Whisper境界: {len(_segs)}セグ → {n}シーン分割')
    except Exception as _we:
        print(f'  ⚠ Whisper境界検出失敗: {_we}')

    # Whisper失敗時はテキスト長比例分割（等分割より大幅に正確）
    if _boundaries is None:
        _lengths = [max(len(t), 1) for t in _cleaned]
        _total_len = sum(_lengths)
        _bds = [0.0]
        _cumul = 0
        for _l in _lengths[:-1]:
            _cumul += _l
            _bds.append(_total * _cumul / _total_len)
        _bds.append(_total)
        _boundaries = _bds
        print(f'  ℹ テキスト長比例分割: {n}シーン')

    # ── 境界でWAVをカット ─────────────────────────────────────────────
    _splits = []
    for _i in range(n):
        _out = tmp_dir / f'el_sp{_i:03d}.wav'
        _ss = _boundaries[_i]
        _ee = _boundaries[_i + 1]
        if _ee - _ss < 0.15:
            _ee = _ss + 0.5
        try:
            ff('-i', str(_full_wav),
               '-af', f'atrim=start={_ss:.4f}:end={_ee:.4f},asetpts=PTS-STARTPTS',
               '-ar', '44100', '-ac', '1', str(_out))
        except Exception as _spe:
            print(f'  ⚠ split{_i}失敗: {_spe}')
            return None
        _splits.append(_out if _out.exists() else None)
    if any(w is None for w in _splits):
        return None
    _avg = (_boundaries[-1] - _boundaries[0]) / n
    print(f'  ✅ 全文一括完了: {_total:.2f}s → {n}シーン (平均{_avg:.2f}s/シーン)')
    return _splits, _avg


def clean_subtitle_text(text):
    """字幕クレンジング: 句読点・冒頭感嘆詞を除去"""
    text = re.sub(r'[\u3002\u3001]', '', text)
    text = re.sub(r'^(\u3048\u3063|\u3042\u3063|\u306d\u3048|\u306d\u3063|\u3046\u308f|\u304a\u3063|\u3078\u3048|\u308f\u3042|\u307e\u3042|\u306f\u3042|\u306f)[\uff01\uff1f!?]?\s*', '', text)
    return text.strip()

def make_sub_cards(text, scene_dur, max_chars=12):
    """
    文節境界優先で1行/最大12文字のカードに分割。
    優先切断: 助詞(は・が・を・に・で・も・て・よ・ね・から・まで)の直後
    次点: 感嘆符・疑問符
    最終手段: max_chars 強制ぶつ切り
    Returns: [(card_text, start_offset, end_offset), ...]
    """
    base = clean_subtitle_text(text)
    base = re.sub(r'#\S+', '', base).strip()
    if not base:
        base = text[:max_chars]
    _BUNSETSU_HIGH = set('はがをにでもてよねへ')
    _KINSOKU_HEAD  = set('！？。、…）」』')

    def _find_break(s, limit):
        """s の [2, limit+2) 範囲内で最適な切断点インデックスを返す"""
        for i in range(min(len(s) - 1, limit + 2), 2, -1):
            c_prev = s[i - 1]
            c_next = s[i] if i < len(s) else ''
            if c_prev in _BUNSETSU_HIGH and c_next not in _KINSOKU_HEAD:
                return i
        for i in range(min(len(s) - 1, limit + 2), 2, -1):
            if s[i - 1] in '！？' and (i >= len(s) or s[i] not in _KINSOKU_HEAD):
                return i
        return limit

    cards = []
    rem = base
    while rem and len(cards) < 4:
        if len(rem) <= max_chars:
            cards.append(rem)
            break
        brk = _find_break(rem, max_chars)
        chunk = rem[:brk].strip()
        if not chunk:
            chunk = rem[:max_chars]
            brk   = max_chars
        cards.append(chunk)
        rem = rem[brk:].strip()
    if not cards:
        cards = [base[:max_chars]]
    while len(cards) > 1 and scene_dur / len(cards) < 0.8:
        cards.pop()
    cards = [re.sub(r'([①-⑧])', r'{\\c&H00FFFFFF&}\1{\\c&H0000FFFF&}', c) for c in cards]
    n = len(cards)
    slot = scene_dur / n
    return [(cards[j], j * slot, (j + 1) * slot - 1.0/30) for j in range(n)]

def validate_sub_cards(cards, scene_text, scene_dur, max_chars=12):
    """字幕カード自己評価: 文字数超過/禁則文頭を検知してClaude修正を1回実行
    cards は make_sub_cards が返す (text, start_off, end_off) のタプルリスト
    """
    _KINSOKU = set('\uff01\uff1f\u3002\u3001\u2026\u300d\uff09\u300b')
    def _strip(t):
        # タプルの場合はテキスト部分のみ取り出す
        s = t[0] if isinstance(t, tuple) else t
        return re.sub(r'\{[^}]*\}', '', str(s))
    def _viol(lst):
        for item in lst:
            raw = _strip(item)
            if len(raw) > max_chars: return f'文字数超過({len(raw)}文字):{raw!r}'
            if raw and raw[0] in _KINSOKU: return f'文頭禁則:{raw!r}'
        return None
    def _rebuild(texts):
        """テキストリストをタプルリストに変換（シーン尺均等配分）"""
        n = len(texts)
        while n > 1 and scene_dur / n < 0.8: n -= 1
        texts = texts[:n]; slot = scene_dur / n
        return [(texts[j], j*slot, (j+1)*slot - 1.0/30) for j in range(n)]
    v = _viol(cards)
    if not v: return cards  # 問題なし
    print(f'    ⚠ 字幕検証NG ({v}) → Claude修正ループ')
    try:
        _raw = call_ai(
            f'以下のセリフを字幕カードに分割してください。\n'
            f'セリフ：{scene_text}\n\n'
            f'【必須ルール】\n'
            f'・1カードは10文字以内\n'
            f'・句読点・記号（！？。、…）を行頭に置かない\n'
            f'・改行で最大3カードに分割\n'
            f'・カードテキストのみ出力（説明不要）',
            tokens=80)
        _lines = [re.sub(r'\{[^}]*\}', '', l.strip()) for l in _raw.strip().split('\n') if l.strip()][:4]
        _lines = [l[:max_chars] for l in _lines if l]
        if _lines and not _viol(_lines):
            print(f'    ✓ 字幕修正完了: {_lines}')
            return _rebuild(_lines)
    except Exception as _ve:
        print(f'    ⚠ Claude修正失敗: {_ve}')
    # 最終手段: タプルからテキストを取り出して強制トリム
    _safe = [_strip(c)[:max_chars] for c in cards]
    return _rebuild(_safe)

def auto_fit_fontsize(text, base=75, min_fs=48):
    """make_sub_cardsで12文字以内に制限済みのため常にbase=75ptを返す"""
    return base


def pre_render_audit(scenes, sub_events, cap_dur, keywords, theme):
    """
    レンダリング前全仕様スキャン。違反はClaude APIで自動修正（Self-Correction Loop）。
    Returns: (sub_events_fixed, all_passed: bool)
    """
    violations = []
    Y_MIN, Y_MAX = int(H * 0.60), int(H * 0.63)

    # 1. 字幕文字数チェック (≤12文字)
    sub_over = []
    for ev in sub_events:
        m = re.search(r'pos\(\d+,\d+\)\\fs\d+\}(.+)$', ev)
        if m:
            txt = re.sub(r'\{[^}]*\}', '', m.group(1))
            if len(txt) > 12:
                sub_over.append(txt)
    if sub_over:
        violations.append(f'字幕文字数超過 {len(sub_over)}件: {sub_over[:3]}')

    # 2. Y座標チェック (H*0.60 ≤ Y ≤ H*0.63)
    y_viols = []
    for ev in sub_events:
        m = re.search(r'\\pos\(\d+,(\d+)\)', ev)
        if m:
            y = int(m.group(1))
            if not (Y_MIN <= y <= Y_MAX):
                y_viols.append(y)
    if y_viols:
        violations.append(f'Y座標違反 {len(y_viols)}件: {sorted(set(y_viols))}')

    # 3. 再生秒数チェック (15s ≤ cap_dur ≤ 58s)
    if not (15.0 <= cap_dur <= 58.0):
        violations.append(f'再生秒数範囲外: {cap_dur:.2f}s (15〜58秒)')

    # 4. ブラックリスト素材チェック
    bl_hits = [kw for kw in keywords if any(b in kw.lower() for b in _KEYWORD_BLACKLIST)]
    if bl_hits:
        violations.append(f'BL素材検知 {len(bl_hits)}件: {bl_hits}')

    if not violations:
        print('  ✅ pre-render Audit: 全4チェック通過')
        return sub_events, True

    print(f'  ⚠ pre-render Audit 違反 ({len(violations)}件) → 自動修正ループ開始')
    for v in violations:
        print(f'    - {v}')

    # Y座標違反: 直接修正（Claudeは不要）
    if y_viols:
        correct_y = str(int(H * 0.62))
        sub_events = [
            re.sub(r'(\\pos\(\d+,)\d+(\))', r'\g<1>' + correct_y + r'\g<2>', ev)
            for ev in sub_events
        ]
        print(f'  ✓ Y座標を{correct_y}に自動修正')

    # 字幕文字数超過: Claude修正ループ
    if sub_over:
        try:
            fixed_evs = []
            for ev in sub_events:
                m = re.search(r'(.*pos\(\d+,\d+\)\\fs\d+\})(.+)$', ev)
                if m:
                    prefix, txt = m.group(1), m.group(2)
                    txt_clean = re.sub(r'\{[^}]*\}', '', txt)
                    if len(txt_clean) > 12:
                        resp = call_ai(
                            f'「{txt_clean}」を10文字以内に要約してください。テキストのみ出力。',
                            tokens=30
                        ).strip()[:12]
                        fixed_evs.append(prefix + resp)
                        continue
                fixed_evs.append(ev)
            sub_events = fixed_evs
            print(f'  ✓ 字幕文字数超過 → Claude修正ループ完了')
        except Exception as _ae:
            print(f'  ⚠ Claude字幕修正失敗（強制トリム）: {_ae}')
            sub_events = [
                re.sub(r'(.*pos\(\d+,\d+\)\\fs\d+\})(.{13,})$',
                       lambda m2: m2.group(1) + m2.group(2)[:12], ev)
                for ev in sub_events
            ]

    return sub_events, len(violations) == 0
def load_bgm_from_drive(duration_sec, out_path, channel_label='yaseru'):
    """DriveのBGMフォルダからMP3をランダム選択してout_pathにWAV変換。なければNone。"""
    import random as _bgmr
    from pathlib import Path as _BP
    _folder = globals().get('BGM_FOLDER', '/content/drive/MyDrive/BGM').rstrip('/')
    _candidates = [f'{_folder}/{channel_label}', _folder]
    _files = []
    for _d in _candidates:
        try:
            _p = _BP(_d)
            _files = [str(x) for ext in ('*.mp3','*.wav','*.m4a') for x in _p.glob(ext)]
            if _files: break
        except Exception: continue
    if not _files: return None
    _chosen = _bgmr.choice(_files)
    try:
        import subprocess as _sp
        _r = _sp.run(
            ['ffmpeg','-y','-stream_loop','-1','-i',_chosen,
             '-t',str(duration_sec+2),'-vn',
             '-acodec','pcm_s16le','-ar','44100','-ac','1',str(out_path)],
            capture_output=True, timeout=60)
        if _r.returncode == 0:
            print(f'  🎵 BGM: {_BP(_chosen).name}')
            return out_path
    except Exception as _be:
        print(f'  ⚠ BGM変換失敗: {_be}')
    return None

def generate_bgm(duration_sec, sr=44100, channel='yaseru'):
    """チャンネル別プロシージャルBGM（ランダムシードで毎回異なる）"""
    import time as _bt
    import os as _btos
    _seed = int.from_bytes(_btos.urandom(4), 'little') % 99999
    n = int(sr * duration_sec)
    bgm = np.zeros(n, dtype=np.float32)
    # yaseru: 128BPM アップテンポ明るい / okane: 96BPM 落ち着いたプロフェッショナル
    if channel == 'okane':
        bpm = 96; beat = 60.0 / bpm
        scale = [261.63, 293.66, 311.13, 349.23, 392.00, 415.30, 466.16, 523.25]
        _patterns = [
            [0, 2, 4, 2, 0, 4, 2, 0, 3, 5, 4, 2, 0, 2, 3, 4],
            [0, 3, 2, 0, 4, 3, 2, 4, 0, 2, 4, 5, 4, 2, 0, 3],
            [2, 0, 4, 2, 5, 4, 2, 0, 3, 2, 0, 4, 5, 3, 2, 0],
        ]
    else:
        bpm = 128; beat = 60.0 / bpm
        scale = [261.63, 293.66, 329.63, 392.00, 440.00, 523.25, 587.33, 659.25]
        _patterns = [
            [0, 2, 4, 5, 4, 2, 0, 2, 4, 7, 6, 5, 4, 2, 0, 4],
            [0, 4, 2, 5, 4, 0, 2, 4, 5, 7, 5, 4, 2, 4, 0, 2],
            [2, 4, 5, 7, 5, 4, 2, 0, 4, 2, 0, 5, 4, 2, 4, 0],
        ]
    rng_p = np.random.default_rng(_seed)
    pattern = _patterns[int(rng_p.integers(0, len(_patterns)))]
    for i in range(int(duration_sec / beat) + len(pattern) + 1):
        pi = i % len(pattern); start = int(i * beat * sr)
        if start >= n: break
        end = min(int((i * beat + beat * 0.75) * sr), n); sz = end - start
        if sz <= 0: continue
        nt = np.linspace(0, sz/sr, sz); freq = scale[pattern[pi] % len(scale)]
        wav = (np.sin(2*np.pi*freq*nt)*0.5 + np.sin(2*np.pi*freq*2*nt)*0.15 + np.sin(2*np.pi*freq*3*nt)*0.08)
        env = np.ones(sz); a, r2 = min(int(0.01*sr), sz), min(int(0.06*sr), sz)
        env[:a] = np.linspace(0, 1, a); env[-r2:] *= np.linspace(1, 0, r2)
        bgm[start:end] += wav * env * 0.22
    for bi in range(int(duration_sec / beat) + 1):
        p = int(bi * beat * sr)
        if p >= n: break
        kl = min(int(0.07*sr), n-p)
        if kl <= 0: continue
        kt = np.linspace(0, 0.07, kl)
        bgm[p:p+kl] += (np.sin(2*np.pi*(90-60*kt/0.07)*kt) * np.exp(-kt*22) * (0.55 if bi%2==0 else 0.22))
    hh_step = beat / 2; rng = np.random.default_rng(_seed + 1)
    for hi in range(int(duration_sec / hh_step) + 1):
        hp = int(hi * hh_step * sr)
        if hp >= n: break
        hl = min(int(0.018*sr), n-hp)
        if hl <= 0: continue
        ht = np.linspace(0, 0.018, hl)
        bgm[hp:hp+hl] += rng.standard_normal(hl) * np.exp(-ht*90) * 0.07
    bass = [130.81, 130.81, 196.00, 146.83]; bar = beat * 4
    for bi in range(int(duration_sec / bar) + 2):
        p = int(bi * bar * sr)
        if p >= n: break
        bl = min(int(bar*0.88*sr), n-p)
        if bl <= 0: continue
        bt = np.linspace(0, bl/sr, bl)
        bgm[p:p+bl] += np.sin(2*np.pi*bass[bi%len(bass)]*bt) * np.exp(-bt*0.8) * 0.38
    mx = np.max(np.abs(bgm))
    if mx > 0: bgm = bgm / mx
    return bgm

def make_cover_crop_vf(iw2, ih2):
    """入力サイズ → 9:16カバークロップ用 ffmpegフィルタ文字列。黒帯ゼロ保証。"""
    # COVER: AR保持のまま両辺がW/H以上になる最小スケール（コンマなし・パース安全）
    if iw2 * H > ih2 * W:       # 横長素材 → 高さ基準でスケール
        sw = iw2 * H // ih2; sh = H
    else:                        # 縦長素材 → 幅基準でスケール
        sw = W; sh = ih2 * W // iw2
    cx2 = (sw - W) // 2; cy2 = (sh - H) // 2  # 中心基準クロップ座標
    return f'scale={sw}:{sh},crop={W}:{H}:{cx2}:{cy2}'

def get_scene_transitions(n_clips, channel='yaseru'):
    """シーン位置・チャンネルに応じてトランジション(type,dur)リストを返す。
    冒頭=引き込み / 中盤=転換インパクト / 解決=安心フェード"""
    _result = []
    for _i in range(n_clips - 1):
        _pos = _i / max(1, n_clips - 2)  # 0.0=最初→1.0=最後
        if _i == 0:
            # 冒頭: フェードで引き込む（いきなり切り替えると混乱）
            _t, _d = 'fade', 0.40
        elif _pos < 0.35:
            # 問題提起フェーズ: 左スライドで緊張感を演出
            _t, _d = 'slideleft', 0.22
        elif _pos < 0.55:
            # 転換点（問題→原因）: クイックワイプでインパクト
            _t, _d = 'wipeleft', 0.18
        elif _pos < 0.80:
            # 解説フェーズ: なめらかなディゾルブ
            _t, _d = 'dissolve', 0.30
        else:
            # 解決策・CTA: ゆっくりフェードで安心感
            _t, _d = 'fade', 0.42
        _result.append((_t, _d))
    return _result

def concat_with_xfade(clips, durations, out, xfade_dur=0.35, transition='fade'):
    """xfadeトランジション結合。transitionはstr(全共通)またはlist of (type,dur)。"""
    n = len(clips)
    if n == 0: return
    if n == 1:
        import shutil as _shu; _shu.copy(str(clips[0]), str(out)); return
    # transition がリストの場合は可変トランジション
    _tlist = transition if isinstance(transition, list) else None
    def _get_t(i):  # i=1..n-1 → (type_str, dur)
        if _tlist and (i-1) < len(_tlist):
            return _tlist[i-1]
        return (transition if isinstance(transition, str) else 'fade'), xfade_dur
    _CHUNK = 15
    if n > _CHUNK:
        _tmp_dir = Path(str(out).replace('.mp4', '_chk'))
        _tmp_dir.mkdir(exist_ok=True)
        chunk_paths, chunk_durs = [], []
        for _ci, _s in enumerate(range(0, n, _CHUNK)):
            _e = min(_s + _CHUNK, n)
            _co = _tmp_dir / f'ck{_ci:03d}.mp4'
            _sd = durations[_s:_e]
            _sub_tlist = _tlist[_s:_e-1] if _tlist else transition
            concat_with_xfade(clips[_s:_e], _sd, _co, xfade_dur, _sub_tlist)
            _chunk_xd = min((_tlist[_s][1] if _tlist and _s < len(_tlist) else xfade_dur), xfade_dur)
            _actual = sum(_sd) - (len(_sd) - 1) * _chunk_xd
            chunk_paths.append(_co)
            chunk_durs.append(_actual + xfade_dur)
        concat_with_xfade(chunk_paths, chunk_durs, out, xfade_dur, 'fade')
        return
    inputs = []
    for c in clips:
        inputs.extend(['-i', str(c)])
    parts = []
    cumulative = 0.0
    label_in = '[0:v]'
    for i in range(1, n):
        _t_str, _xd = _get_t(i)
        cumulative += durations[i-1] - _xd
        label_out = '[vout]' if i == n - 1 else f'[x{i}]'
        parts.append(
            f'{label_in}[{i}:v]xfade=transition={_t_str}:'
            f'duration={_xd:.3f}:offset={cumulative:.4f}{label_out}'
        )
        label_in = label_out
    fc = ';'.join(parts)
    _enc = _get_encoder()
    if _enc == 'h264_nvenc':
        ff(*inputs, '-filter_complex', fc, '-map', '[vout]',
           '-c:v', 'h264_nvenc', '-preset', 'p4', '-cq', '20',
           '-pix_fmt', 'yuv420p', str(out))
    else:
        ff(*inputs, '-filter_complex', fc, '-map', '[vout]',
           '-c:v', 'libx264', '-preset', 'fast', '-crf', '22',
           '-threads', '0', '-pix_fmt', 'yuv420p', str(out))

def make_clip(img_path, dur, out, idx=0, boost=1.0, static_zoom=None):
    """静止画クリップ: 動的Ken Burns or 静的センターズーム。
    static_zoom: float指定時(例:1.25)は静止画を1/zoom倍クロップ→スケールアップ（ジャンプカット後半用）
    """
    d = max(dur, 0.1)
    N = f"{d * FPS:.1f}"
    ZS = f"{1.00 * boost:.4f}"  # zoom-in 開始倍率（等倍スタート）
    ZE = f"{1.14 * boost:.4f}"  # zoom-in 終了倍率（緩やかに）
    ZD = f"{0.14 * boost:.4f}"  # ズーム変化幅（小さく → 自然な動き）
    # n=出力フレーム番号。コンマ不使用でffmpegフィルタチェーンのパースバグを回避
    styles = [
        # 0: zoom-in(1.05→1.25) + 右パン
        (f"iw/({ZS}+{ZD}*n/{N})", f"ih/({ZS}+{ZD}*n/{N})",
         f"(iw-iw/({ZS}+{ZD}*n/{N}))*n/{N}", f"(ih-ih/({ZS}+{ZD}*n/{N}))/2"),
        # 1: zoom-in(1.05→1.25) + 下パン
        (f"iw/({ZS}+{ZD}*n/{N})", f"ih/({ZS}+{ZD}*n/{N})",
         f"(iw-iw/({ZS}+{ZD}*n/{N}))/2", f"(ih-ih/({ZS}+{ZD}*n/{N}))*n/{N}"),
        # 2: zoom-out(1.25→1.05) + 左パン
        (f"iw/({ZE}-{ZD}*n/{N})", f"ih/({ZE}-{ZD}*n/{N})",
         f"(iw-iw/({ZE}-{ZD}*n/{N}))*(1-n/{N})", f"(ih-ih/({ZE}-{ZD}*n/{N}))/2"),
        # 3: zoom-out(1.25→1.05) + 上パン
        (f"iw/({ZE}-{ZD}*n/{N})", f"ih/({ZE}-{ZD}*n/{N})",
         f"(iw-iw/({ZE}-{ZD}*n/{N}))/2", f"(ih-ih/({ZE}-{ZD}*n/{N}))*(1-n/{N})"),
    ]
    _style_idx = idx % len(styles)  # 連続する方向で自然な流れ
    cw, ch, cx, cy = styles[_style_idx]
    if static_zoom and Path(img_path).exists():
        # 静的センターズーム: 1/static_zoom クロップ → W x H スケールアップ
        try:
            _pb2 = subprocess.run(['ffprobe','-v','quiet','-print_format','json','-show_streams',str(img_path)],
                                  capture_output=True, text=True, timeout=10)
            _vs2 = next((s for s in _json.loads(_pb2.stdout).get('streams',[])
                         if s.get('codec_type')=='video'), None)
            _iw2, _ih2 = (int(_vs2['width']), int(_vs2['height'])) if _vs2 else (W, H)
        except Exception:
            _iw2, _ih2 = W, H
        cover2 = make_cover_crop_vf(_iw2, _ih2)
        _zw = int(W / static_zoom); _zh = int(H / static_zoom)
        _zx = (W - _zw) // 2;       _zy = (H - _zh) // 2
        vf_sz = f"{cover2},scale={W}:{H},crop={_zw}:{_zh}:{_zx}:{_zy},scale={W}:{H},{_CINEMA_VF}"
        _enc2 = _get_encoder()
        if _enc2 == 'h264_nvenc':
            ff('-loop','1','-i',str(img_path),'-vf',vf_sz,
               '-t',str(dur),'-r',str(FPS),'-an',
               '-c:v','h264_nvenc','-preset','p4','-cq','20','-pix_fmt','yuv420p',str(out))
        else:
            ff('-loop','1','-i',str(img_path),'-vf',vf_sz,
               '-t',str(dur),'-r',str(FPS),'-an',
               '-c:v','libx264','-preset','fast','-crf','23','-threads','0','-pix_fmt','yuv420p',str(out))
        return
    if Path(img_path).exists():
        # 9:16カバークロップ: 画像の実寸をプローブして黒帯ゼロを保証
        try:
            _pb = subprocess.run(['ffprobe','-v','quiet','-print_format','json','-show_streams',str(img_path)],
                                 capture_output=True, text=True, timeout=10)
            _vs = next((s for s in _json.loads(_pb.stdout).get('streams',[])
                        if s.get('codec_type')=='video'), None)
            _iw, _ih = (int(_vs['width']), int(_vs['height'])) if _vs else (W, H)
        except:
            _iw, _ih = W, H
        cover = make_cover_crop_vf(_iw, _ih)
        # センタークロップ → Ken Burns → 映像覚醒フィルター（1コマンドチェーン）
        vf = f"{cover},crop={cw}:{ch}:{cx}:{cy},scale={W}:{H},{_CINEMA_VF}"
        _enc = _get_encoder()
        if _enc == 'h264_nvenc':
            ff('-loop','1','-i',str(img_path),'-vf',vf,
               '-t',str(dur),'-r',str(FPS),'-an',
               '-c:v','h264_nvenc','-preset','p4','-cq','20',
               '-pix_fmt','yuv420p',str(out))
        else:
            ff('-loop','1','-i',str(img_path),'-vf',vf,
               '-t',str(dur),'-r',str(FPS),'-an',
               '-c:v','libx264','-preset','fast','-crf','23',
               '-threads','0','-pix_fmt','yuv420p',str(out))
    else:
        ff('-f','lavfi','-i',f'color=c=black:s={W}x{H}:r={FPS}',
           '-t',str(dur),'-vf',_CINEMA_VF,
           '-c:v','libx264','-preset','ultrafast','-threads','0',str(out))

def make_gradient_bg(out_path, scene_idx=0, text=''):
    """Pexels取得失敗時の代替: シーンごとに色が変わるグラデーション背景+テキスト"""
    _GRADIENT_USED[0] = True  # グラデBG使用を記録（アップロードスキップ判定に使用）
    # BGR: 6種類のカラーテーマ（scene_idxで循環）
    _themes = [
        ([77,  0, 38], [80,  0,  0]),   # パープル→ダークレッド
        ([0,  60, 20], [0,  80, 40]),   # ダークグリーン→エメラルド
        ([80, 40,  0], [60, 20,  0]),   # ダークオレンジ→ブラウン
        ([0,  40, 80], [0,  20, 60]),   # ディープブルー→ネイビー
        ([60,  0, 60], [30,  0, 80]),   # ダークマゼンタ→インディゴ
        ([0,  60, 60], [0,  40, 40]),   # ダークティール→シアン
    ]
    c1_l, c2_l = _themes[scene_idx % len(_themes)]
    c1 = np.array(c1_l, dtype=np.float32)
    c2 = np.array(c2_l, dtype=np.float32)
    img = np.zeros((H, W, 3), dtype=np.uint8)
    for y in range(H):
        t = y / H
        img[y, :] = (c1 + (c2 - c1) * t).astype(np.uint8)
    cv2.imwrite(str(out_path), img)
    if text:
        try:
            from PIL import Image as _PILImage, ImageDraw as _Draw, ImageFont as _Font
            pil_img = _PILImage.open(str(out_path)).convert('RGB')
            draw = _Draw.Draw(pil_img)
            _font_path = '/usr/share/fonts/opentype/noto/NotoSansCJK-Bold.ttc'
            try:
                font = _Font.truetype(_font_path, 72)
                font_sm = _Font.truetype(_font_path, 48)
            except Exception:
                font = _Font.load_default()
                font_sm = font
            # シーン番号
            draw.text((60, 80), f'Scene {scene_idx + 1}', font=font_sm, fill=(200, 200, 200))
            # セリフテキスト（折り返し）
            _max_w = W - 120
            _chars_per_line = max(10, _max_w // 72)
            _lines = []
            _remaining = text
            while _remaining:
                _lines.append(_remaining[:_chars_per_line])
                _remaining = _remaining[_chars_per_line:]
            _y = H // 2 - len(_lines) * 80
            for _line in _lines:
                bbox = draw.textbbox((0, 0), _line, font=font)
                _lw = bbox[2] - bbox[0]
                _x = (W - _lw) // 2
                draw.text((_x + 3, _y + 3), _line, font=font, fill=(0, 0, 0))
                draw.text((_x, _y), _line, font=font, fill=(255, 255, 255))
                _y += 90
            pil_img.save(str(out_path))
        except Exception as _pe:
            pass

def fetch_pexels_video(query, out_path, scene_idx=0, duration_target=4.0):
    """Pexels Videos APIから関連動画クリップを取得"""
    _pk = globals().get('PEXELS_API_KEY', '').strip()
    if not _pk:
        return None
    try:
        import requests as _rv, random as _rvr
        _page = _rvr.randint(1, 3)
        _r = _rv.get('https://api.pexels.com/videos/search',
                     headers={'Authorization': _pk},
                     params={'query': query, 'per_page': 10, 'page': _page,
                             'orientation': 'portrait', 'size': 'medium'},
                     timeout=15)
        if _r.status_code != 200:
            return None
        _videos = _r.json().get('videos', [])
        # 未使用・縦向きを優先
        for _v in _videos:
            if _v['id'] in _USED_PEXELS_VIDEO_IDS:
                continue
            # 動画ファイルを選択（HD優先、縦向き優先）
            _files = sorted(_v.get('video_files', []),
                           key=lambda f: (f.get('height', 0) >= 720,
                                         f.get('width', 9999) < f.get('height', 0),
                                         f.get('height', 0)), reverse=True)
            for _vf in _files:
                if _vf.get('width', 0) and _vf.get('height', 0):
                    _dl = _rv.get(_vf['link'], timeout=30, stream=True)
                    if _dl.status_code == 200:
                        with open(str(out_path), 'wb') as _of:
                            for _chunk in _dl.iter_content(8192):
                                _of.write(_chunk)
                        _USED_PEXELS_VIDEO_IDS.add(_v['id'])
                        return out_path
    except Exception as _ve:
        print(f'    Pexels動画取得失敗: {_ve}')
    return None


def fetch_pexels_image_robust(kw, img_fn, scene_idx=0, scene_text=''):
    """1KW 1回検索: portrait固定・401は10秒待ちリトライ・グラデBGは最終手段"""
    # ── ハードコードBLガード（検索前に禁止KW強制除去）─────────────────
    _FETCH_BL = {'revenue','report','graph','chart','business','office',
                 'money','pc','tech','data','spreadsheet','corporate','laptop'}
    if any(b in kw.lower() for b in _FETCH_BL):
        _safe = 'healthy food'
        print(f'    🚫 検索BL: {kw!r} → {_safe!r}')
        kw = _safe
    _pkey = ''.join(c for c in (PEXELS_API_KEY or '') if 0x21 <= ord(c) <= 0x7E)
    if not _pkey:
        print('  ⚠ PEXELS_API_KEY未設定 → グラデBG')
        make_gradient_bg(img_fn, scene_idx, scene_text)
        return

    def _has_jp(s):
        return bool(re.search(r'[\u3040-\u309F\u30A0-\u30FF\u4E00-\u9FFF]', s))

    def _try_fetch(query):
        """1キーワードで写真取得を試みる。成功したらTrueを返す"""
        _pexels_wait()
        try:
            import random as _pex_rand
            _pg = _pex_rand.randint(1, 5)  # ランダムページで多様化・重複排除
            params = {'query': query, 'per_page': 10, 'orientation': 'portrait', 'page': _pg}
            r = _req.get('https://api.pexels.com/v1/search',
                         headers={'Authorization': _pkey},
                         params=params, timeout=15)
            print(f'    [img] {query!r} HTTP={r.status_code}', end='')
            if r.status_code == 401:
                print(' ← レート制限? 10秒待機...', end='')
                time.sleep(10)
                _PEXELS_LAST_T[0] = 0.0  # 次の_pexels_waitをすぐ通過
                _pexels_wait()
                r = _req.get('https://api.pexels.com/v1/search',
                             headers={'Authorization': _pkey},
                             params=params, timeout=15)
                print(f' retry HTTP={r.status_code}', end='')
            if r.status_code == 429:
                print(' ← レート制限 30秒待機...', end='')
                time.sleep(30)
                _PEXELS_LAST_T[0] = 0.0
                return False
            if r.status_code != 200:
                print(' ← スキップ')
                return False
            photos = r.json().get('photos', [])
            if not photos:
                print(' ← 0件')
                return False
            # 重複排除: 未使用の写真を選択
            _selected = None
            for _ph in photos:
                if _ph['id'] not in _USED_PEXELS_IDS:
                    _selected = _ph
                    break
            if _selected is None:
                _selected = photos[0]  # 全て使用済みなら先頭を再利用
            url = (_selected['src'].get('portrait')
                   or _selected['src'].get('large2x')
                   or _selected['src'].get('large'))
            if not url:
                print(' ← URL取得失敗')
                return False
            dl = _req.get(url, timeout=30)
            if dl.status_code == 200 and len(dl.content) > 2048:
                arr = np.frombuffer(dl.content, np.uint8)
                img_check = cv2.imdecode(arr, cv2.IMREAD_COLOR)
                if img_check is not None:
                    img_fn.write_bytes(dl.content)
                    _USED_PEXELS_IDS.add(_selected['id'])
                    print(f' ✓ ({len(dl.content)//1024}KB)')
                    return True
            print(f' ← DL失敗')
            return False
        except Exception as _e:
            print(f' ← 例外: {_e}')
            return False

    kw_clean = kw.strip()
    # キーワード候補リスト（最大3つ: 元KW → 先頭1語 → 汎用）
    _candidates = []
    if not _has_jp(kw_clean):
        _candidates.append(kw_clean)
        _fw = kw_clean.split()[0]
        if _fw != kw_clean and not _has_jp(_fw):
            _candidates.append(_fw)
    _candidates.append('lifestyle healthy')  # 汎用フォールバック

    for _q in _candidates:
        if _try_fetch(_q):
            return
    print(f'  ⚠ 全キーワード失敗 → グラデーション背景')
    make_gradient_bg(img_fn, scene_idx, scene_text)


# ── Gemini画像生成: 利用可能モデルをキャッシュ ─────────────────────
_GEMINI_IMG_MODEL = [None]  # [0]=検出済みモデル名 or 'NONE'
_GEMINI_CALL_COUNTER = [0]  # 1動画あたりのGemini呼び出し回数（上限3）
_GRADIENT_USED = [False]    # グラデBG使用フラグ（動画生成ごとにリセット）
_GRADIENT_SKIP_PATHS = set()  # グラデBG混入 → アップロードスキップ対象パス
_USED_PEXELS_IDS = set()  # Pexels重複排除: 使用済み写真ID
_USED_PEXELS_VIDEO_IDS = set()  # Pexels動画重複排除: 使用済み動画ID

def _detect_gemini_image_model(api_key):
    """利用可能な画像生成モデルを自動検出してキャッシュ"""
    # 手動指定が優先
    _manual = globals().get('GEMINI_IMAGE_MODEL', '').strip()
    if _manual:
        if _GEMINI_IMG_MODEL[0] != _manual:
            print(f'  Geminiモデル（手動指定）: {_manual}')
            _GEMINI_IMG_MODEL[0] = _manual
        return _manual
    if _GEMINI_IMG_MODEL[0] is not None:
        return _GEMINI_IMG_MODEL[0]
    from google import genai
    # ハードコード優先順リスト（テキストモデルは含めない）
    _hardcoded = [
        'gemini-2.0-flash-preview-image-generation',
        'gemini-2.0-flash-exp-image-generation',
        'gemini-2.0-flash-image-generation',
    ]
    try:
        _client = genai.Client(api_key=api_key)
        _all_models = list(_client.models.list())
        _available = {m.name.split('/')[-1] for m in _all_models}
        # 'image'または'imagen'を含むモデルのみ抽出（テキストモデルを除外）
        _img_models = sorted(
            n for n in _available
            if ('image' in n.lower() or 'imagen' in n.lower())
            and 'gemini' in n.lower()
        )
        print(f'  Geminiモデル全体: {len(_available)} / 画像生成対応: {len(_img_models)}')
        if _img_models:
            print(f'  利用可能な画像モデル: {_img_models}')
        # ① ハードコードリストで一致するものを優先
        for _c in _hardcoded:
            if _c in _available:
                print(f'  Gemini画像モデル確定（既知）: {_c}')
                _GEMINI_IMG_MODEL[0] = _c
                return _c
        # ② models.list()で検出された画像モデルを使う（新モデル対応）
        if _img_models:
            _chosen = _img_models[0]
            print(f'  Gemini画像モデル確定（自動検出）: {_chosen}')
            _GEMINI_IMG_MODEL[0] = _chosen
            return _chosen
        print(f'  画像生成対応モデルなし → PROBE')
    except Exception as _le:
        print(f'  モデルリスト取得失敗: {_le}')
    _GEMINI_IMG_MODEL[0] = 'PROBE'
    return 'PROBE'

def fetch_pollinations_image(prompt, img_fn, scene_idx=0):
    """Pollinations AI: 完全無料・登録不要のAI画像生成フォールバック"""
    try:
        import urllib.parse, requests as _req_pol
        from io import BytesIO
        from PIL import Image as _PILImg
        _enc = urllib.parse.quote(
            f'vertical portrait 9:16, {prompt}, no text, no watermarks, clean aesthetic'
        )
        _url = (f'https://image.pollinations.ai/prompt/{_enc}'
                f'?width=540&height=960&seed={scene_idx}&enhance=true&nologo=true')
        _r = _req_pol.get(_url, timeout=30)
        if _r.status_code == 200 and len(_r.content) > 5000:
            _img = _PILImg.open(BytesIO(_r.content)).resize((W, H), _PILImg.LANCZOS)
            _img.save(str(img_fn), quality=95)
            print(' OK Pollinations生成（無料AI）')
            return True
        print(f'    Pollinations: HTTP={_r.status_code}')
    except Exception as _pe:
        print(f'    Pollinations失敗: {_pe}')
    return False


def fetch_mascot_image(img_fn, scene_idx=0):
    """マスコット（白猫）をGeminiで生成 → Pollinations フォールバック"""
    _prompt = globals().get('MASCOT_PROMPT', '可愛い白猫キャラクター シンプルフラットイラスト')
    _gkey   = globals().get('GOOGLE_API_KEY', '')
    if _gkey and _GEMINI_IMG_MODEL[0] != 'NONE':
        try:
            from google import genai
            from google.genai import types
            from io import BytesIO
            from PIL import Image as _PILImg
            _fp = (
                f'Vertical 9:16 portrait. Flat digital illustration. '
                f'{_prompt}. Soft pastel background, health & wellness mood. '
                f'No text, no watermarks, centered composition.'
            )
            _result = genai.Client(api_key=_gkey).models.generate_images(
                model='imagen-3.0-generate-002',
                prompt=_fp,
                config=types.GenerateImagesConfig(
                    number_of_images=1,
                    output_mime_type='image/jpeg',
                    aspect_ratio='9:16',
                ),
            )
            for _gi in _result.generated_images:
                _img = _PILImg.open(BytesIO(_gi.image.image_bytes)).resize((W, H), _PILImg.LANCZOS)
                _img.save(str(img_fn), quality=95)
                print(' ✓ マスコット(白猫)生成')
                return
        except Exception as _me:
            print(f'    ⚠ マスコット生成失敗: {_me} → Pollinations')
    if not fetch_pollinations_image(_prompt, img_fn, scene_idx):
        make_gradient_bg(img_fn, scene_idx, '')


def make_asmr_tick(sr=44100):
    """診断項目切り替え時の短い「チン」音(50ms)を生成してWAVバイト列で返す"""
    import wave, io
    dur = 0.05
    n   = int(sr * dur)
    t   = np.linspace(0, dur, n, endpoint=False)
    wave_data = (np.sin(2 * np.pi * 900 * t) * np.exp(-t * 100) * 0.25 * 32767).astype(np.int16)
    buf = io.BytesIO()
    with wave.open(buf, 'wb') as wf:
        wf.setnchannels(1); wf.setsampwidth(2); wf.setframerate(sr)
        wf.writeframes(wave_data.tobytes())
    return buf.getvalue()

def fetch_gemini_image(prompt, img_fn, scene_idx=0):
    # ── 上限ガード: 1動画あたり最大3回まで ──────────────────────
    _GEMINI_CALL_COUNTER[0] += 1
    if _GEMINI_CALL_COUNTER[0] > 3:
        raise RuntimeError(
            f'❌ Gemini API上限超過: {_GEMINI_CALL_COUNTER[0]}回目の呼び出し（上限3回）。'
            'コードのバグを確認してください。'
        )
    print(f'    [Gemini #{_GEMINI_CALL_COUNTER[0]}/3] ', end='')
    _gkey = globals().get('GOOGLE_API_KEY', '')
    _style_map = {
        'realistic':    'photorealistic, cinematic soft lighting, high quality',
        'anime':        'anime art style, vibrant colors, clean',
        'manga':        'manga style, black and white line art',
        'illustration': 'flat digital illustration, colorful, modern',
    }
    _style = _style_map.get(globals().get('IMAGE_STYLE', 'realistic'), 'photorealistic')
    _fp = (f'Vertical 9:16 portrait format. {_style}. {prompt}. '
           'No text overlays, no watermarks, no logos. '
           'Clean aesthetic, soft focus background, wellness mood.')

    # ── ① Imagen 3 (generate_images API) ────────────────────────
    if _gkey and _GEMINI_IMG_MODEL[0] != 'NONE':
        try:
            from google import genai
            from google.genai import types
            from io import BytesIO
            from PIL import Image as _PILImg
            _client = genai.Client(api_key=_gkey)
            _result = _client.models.generate_images(
                model='imagen-3.0-generate-002',
                prompt=_fp,
                config=types.GenerateImagesConfig(
                    number_of_images=1,
                    output_mime_type='image/jpeg',
                    aspect_ratio='9:16',
                ),
            )
            for _gi in _result.generated_images:
                _img = _PILImg.open(BytesIO(_gi.image.image_bytes))
                _img = _img.resize((W, H), _PILImg.LANCZOS)
                _img.save(str(img_fn), quality=95)
                print(' OK Imagen3生成')
                return
            print('    Imagen3: 画像なし')
        except Exception as _e:
            _emsg = str(_e)
            print(f'    Imagen3失敗: {type(_e).__name__}: {_emsg[:120]}')
            if '403' in _emsg or 'PERMISSION_DENIED' in _emsg or 'billing' in _emsg.lower():
                print('    → 課金プロジェクトのAPIキーが必要です（profound-jet-497410-d6 でキー再発行）')
                _GEMINI_IMG_MODEL[0] = 'NONE'

    # ── ② Gemini Flash generate_content API（無料・モデル名を動的検出）──
    if _gkey:
        from google import genai as _genai_f
        from google.genai import types as _types_f
        _fc_f = _genai_f.Client(api_key=_gkey)
        # ハードコードリスト（新旧モデル名を網羅）
        _flash_models = [
            'gemini-2.0-flash-exp-image-generation',
            'gemini-2.0-flash-preview-image-generation',
            'gemini-2.0-flash-exp',
            'gemini-2.0-flash',
            'gemini-1.5-flash',
        ]
        # models.list() で現在有効な画像モデルを先頭に追加
        try:
            _live = [m.name.split('/')[-1] for m in _fc_f.models.list()
                     if 'image' in m.name.lower() or 'flash' in m.name.lower()]
            _flash_models = _live + [m for m in _flash_models if m not in _live]
        except Exception:
            pass
        for _fm in _flash_models:
            try:
                _fr = _fc_f.models.generate_content(
                    model=_fm,
                    contents=_fp,
                    config=_types_f.GenerateContentConfig(
                        response_modalities=['IMAGE', 'TEXT']
                    )
                )
                _got_img = False
                for _part in _fr.candidates[0].content.parts:
                    if hasattr(_part, 'inline_data') and _part.inline_data is not None:
                        from PIL import Image as _PILF
                        import io as _ioF
                        _img_f = _PILF.open(_ioF.BytesIO(_part.inline_data.data))
                        _img_f = _img_f.convert('RGB').resize((1080, 1920), _PILF.LANCZOS)
                        _img_f.save(str(img_fn), quality=95)
                        print(f' OK Gemini Flash({_fm})')
                        return
                if not _got_img:
                    # 画像が返らなかった（テキストのみ）→ 次のモデル
                    continue
            except Exception as _fe:
                _fmsg = str(_fe)
                if any(x in _fmsg for x in ['404', 'not found', 'NOT_FOUND', 'does not support']):
                    continue  # モデル名が違う → 次を試す
                if '429' in _fmsg:
                    print(f'    Flash: レート制限 → Pollinationsへ')
                    break
                if any(x in _fmsg for x in ['400', 'INVALID', 'not support image']):
                    continue  # 画像生成非対応モデル → 次を試す
                print(f'    Flash({_fm}): {_fmsg[:80]}')
                break

    # ── ③ Pollinations AI（完全無料フォールバック）───────────────
    print('    Pollinations AIにフォールバック... ', end='')
    if fetch_pollinations_image(prompt, img_fn, scene_idx):
        return

    # ── ③ Pexels ────────────────────────────────────────────────
    _pkey_chk = ''.join(c for c in (globals().get('PEXELS_API_KEY','') or '') if 0x21 <= ord(c) <= 0x7E)
    if _pkey_chk:
        print('    Pexelsにフォールバック')
        fetch_pexels_image_robust(prompt, img_fn, scene_idx, '')
        return

    # ── ④ グラデBG（最終手段）────────────────────────────────────
    print('    グラデBG')
    make_gradient_bg(img_fn, scene_idx, '')
def make_clip_from_video(video_path, dur, out, idx=0, boost=1.0):
    """Pexels動画を縦型1080x1920にクロップ・ループ変換（動的 Ken Burns 付き）"""
    probe = subprocess.run(['ffprobe','-v','quiet','-print_format','json','-show_streams',str(video_path)],
                           capture_output=True, text=True)
    try:
        vs = next((s for s in _json.loads(probe.stdout).get('streams',[]) if s.get('codec_type')=='video'), None)
        vw, vh = (int(vs['width']), int(vs['height'])) if vs else (W, H)
    except:
        vw, vh = W, H
    cover = make_cover_crop_vf(vw, vh)  # 9:16カバークロップ（黒帯ゼロ・中心基準）
    d = max(dur, 0.1)
    N = f"{d * FPS:.1f}"
    ZS = f"{1.00 * boost:.4f}"  # zoom-in 開始倍率（等倍スタート）
    ZE = f"{1.14 * boost:.4f}"  # zoom-in 終了倍率（緩やかに）
    ZD = f"{0.14 * boost:.4f}"  # ズーム変化幅（小さく → 自然な動き）
    # n=出力フレーム番号。コンマ不使用でffmpegフィルタチェーンのパースバグを回避
    kb = [
        # zoom-in(1.05→1.25) + 右パン
        (f"iw/({ZS}+{ZD}*n/{N})", f"ih/({ZS}+{ZD}*n/{N})",
         f"(iw-iw/({ZS}+{ZD}*n/{N}))*n/{N}", f"(ih-ih/({ZS}+{ZD}*n/{N}))/2"),
        # zoom-out(1.25→1.05) + 左パン
        (f"iw/({ZE}-{ZD}*n/{N})", f"ih/({ZE}-{ZD}*n/{N})",
         f"(iw-iw/({ZE}-{ZD}*n/{N}))*(1-n/{N})", f"(ih-ih/({ZE}-{ZD}*n/{N}))/2"),
    ]
    kcw, kch, kcx, kcy = kb[idx % len(kb)]
    ff('-stream_loop','-1','-i',str(video_path),
       '-vf',f'{cover},crop={kcw}:{kch}:{kcx}:{kcy},scale={W}:{H}',
       '-t',str(dur),'-r',str(FPS),'-an',
       '-c:v','libx264','-preset','fast','-crf','23','-pix_fmt','yuv420p',str(out))


# ── ジャンルブラックリスト（テーマ無関係素材の混入を100%防ぐ）────────
_KEYWORD_BLACKLIST = {
    # IT / Tech / Data
    'data','graph','chart','spreadsheet','code','server','network','laptop',
    'computer','keyboard','monitor','dashboard','analytics','database',
    'algorithm','software','programming','cloud','cyber','digital screen',
    # Finance / Business / Office
    'revenue','profit','stock','market','investment','finance','meeting',
    'office','boardroom','presentation','report','business','corporate',
    'dollar','money chart','financial','budget','sales','workspace','desk',
    # Generic / Low-visual-quality
    'lifestyle','person','woman','man','people','human',
}

# ── ダイエット専用 許可KWリスト ──────────────────────────────────
_DIET_ALLOWED_KEYWORDS = [
    'diet', 'healthy food', 'morning', 'warm water',
    'stretching', 'weight scale', 'sleeping',
]

# ── 意味・映像完全同期マッピング表（台本行 → Pexels検索クエリ を1:1固定）────
_SEMANTIC_MAP = [
    # タンパク質・食材系
    (['タンパク質','プロテイン','鶏肉','肉','卵','魚','チキン'],
     ['chicken breast', 'eggs protein', 'grilled meat']),
    # 夜・時間系
    (['夜8時','夜間','夜の','深夜','夜中','夜より','夜は'],
     ['clock evening', 'night room lamp']),
    # ダイエット成果・体型系
    (['体が変わる','痩せた','体重','ダイエット成功','体型','スリム'],
     ['weight scale', 'diet female fit']),
    # 代謝・脂肪燃焼系
    (['代謝','燃える','脂肪','脂質','大チャンス','爆上がり'],
     ['body fitness', 'healthy metabolism']),
    # 食事・食べる系
    (['食べながら','食べる','食事','ご飯','料理'],
     ['healthy meal', 'diet food plate']),
    # 飲み物・白湯系
    (['白湯','水を','飲む','ドリンク'],
     ['warm water cup', 'drinking water']),
    # 朝・習慣系
    (['朝より','朝の','朝より','モーニング','起きたら'],
     ['morning routine', 'morning healthy']),
    # 運動・ストレッチ系
    (['運動','ストレッチ','筋トレ','ヨガ'],
     ['stretching exercise', 'home workout']),
    # 睡眠系
    (['睡眠','寝る','眠り'],
     ['sleeping bedroom', 'night rest']),
    # フック系（曖昧）
    (['マジ','やばい','知ってた','秘密','え、'],
     ['healthy food', 'diet']),
    # アウトロ・CTA系
    (['保存して','試してね','試してみて','やらない','ないよね','挑戦'],
     ['healthy lifestyle', 'morning routine']),
]

def get_semantic_query(line, theme):
    """台本の1行テキストから意味一致するPexels検索クエリを決定（静的マップ優先）"""
    import random as _sr
    for kws, queries in _SEMANTIC_MAP:
        if any(kw in line for kw in kws):
            return _sr.choice(queries)
    if any(k in theme for k in ['ダイエット', '痩せ', '健康', '体重', '脂肪']):
        return _sr.choice(_DIET_ALLOWED_KEYWORDS)
    return 'healthy food'

def ai_batch_visual_queries(scene_texts, theme):
    """全シーンのvisualクエリをClaude AIで一括生成。静的マップより正確。"""
    if not scene_texts or not globals().get('CLAUDE_API_KEY', '').strip():
        return None
    _lines = '\n'.join(f'{i+1}. {t}' for i, t in enumerate(scene_texts))
    _prompt = (
        f'YouTube Shorts「{theme}」の映像クエリを生成してください。\n\n'
        f'台本（各行=1シーン）:\n{_lines}\n\n'
        f'各シーンに対し、Pexels映像検索に使う英語2〜3語のクエリを出力してください。\n'
        f'厳守ルール:\n'
        f'・シーンの内容・時間帯・感情を正確に反映する\n'
        f'・夜→night dark / 朝→morning bright / 脳・考える→thinking brain\n'
        f'・疲れ→tired exhausted / 食べたい→craving food night\n'
        f'・具体的な食材→protein shake / chicken meal / eggs breakfast\n'
        f'・抽象語は視覚化: 代謝→body fitness glow / 意志→determined person\n'
        f'・business/office/computer/chart/graph は絶対禁止\n'
        f'・ホラー・暴力・ギャンブル・ビットコイン・仮想通貨コインの映像は絶対禁止\n'
        f'・金融/投資コンテンツの場合: japanese person phone / savings piggy bank / '
        f'yen money / growth chart upward / budget notebook / investment book\n'
        f'・金融コンテンツで絶対使わない: bitcoin / crypto coin / casino / dice / horror / '
        f'foreign man / caucasian businessman\n\n'
        f'出力形式（番号なし・1行1クエリ・英語のみ）:\n'
        f'night kitchen dark\n'
        f'tired brain thinking\n'
        f'（以降同様に{len(scene_texts)}行）'
    )
    try:
        _raw = call_ai(_prompt, tokens=len(scene_texts) * 12 + 50)
        _qs = [l.strip().lower() for l in _raw.strip().split('\n')
               if l.strip() and not l.strip()[0].isdigit()]
        if len(_qs) >= len(scene_texts):
            return _qs[:len(scene_texts)]
        # 足りない分は末尾をパディング
        while len(_qs) < len(scene_texts):
            _qs.append(_qs[-1] if _qs else 'healthy food')
        return _qs
    except Exception as _e:
        print(f'  ⚠ AI visual query失敗: {_e}')
        return None

def assert_semantic_query(query, line):
    """検索クエリのBL違反・意味不整合をAssertで100%遮断"""
    _QBL = {'lifestyle','revenue','report',
             'graph','chart','business','office','computer','tech','desktop',
             'bitcoin','crypto','horror','casino','gambling','dice',
             'coin','skull','blood','weapon','violence','nude'}
    q_lower = query.lower().strip()
    for bad in _QBL:
        if bad == q_lower or f' {bad}' in q_lower or f'{bad} ' in q_lower:
            raise AssertionError(
                f'SemanticAssert BL違反: query={query!r} に禁止語「{bad}」 (line:{line!r})')


def sanitize_visual_keyword(kw: str, theme: str) -> str:
    """ブラックリスト語を含むキーワードをテーマ合致の安全なものへ置換"""
    kw_lower = kw.lower()
    for bad in _KEYWORD_BLACKLIST:
        if bad in kw_lower:
            try:
                safe = call_ai(
                    f'テーマ「{theme}」の動画シーンに合う英語の映像キーワードを1〜3語で答えてください。\n'
                    f'例: healthy food / morning routine / fresh vegetables\n'
                    f'キーワードのみ出力（説明不要）',
                    tokens=30
                ).strip().lower()
                if any(b in safe for b in _KEYWORD_BLACKLIST):
                    safe = 'healthy food'
                print(f'    🚫 BL置換: {kw!r} → {safe!r}')
                return safe
            except Exception:
                pass
            return 'healthy food'
    return kw
W, H, FPS = 1080, 1920, 30

# ── 映像覚醒フィルター: シネマグレーディング + アンシャープマスク ────
# unsharp: 輪郭強調（シズル感・湯気・ディテール）
# eq: 彩度1.25倍 + コントラスト微増
# curves: シャドウに青みを足してシネマティックに / ハイライトはクリア
_CINEMA_VF = (
    'unsharp=luma_msize_x=5:luma_msize_y=5:luma_amount=0.8'
    ':chroma_msize_x=3:chroma_msize_y=3:chroma_amount=0.4,'
    'eq=saturation=1.25:contrast=1.03,'
    "curves="
    "r='0/0 0.08/0.06 0.5/0.5 0.9/0.95 1/1':"
    "g='0/0 0.08/0.07 0.5/0.5 0.9/0.93 1/1':"
    "b='0/0.04 0.12/0.15 0.5/0.52 0.9/0.93 1/0.98'"
)

# ── ハードウェアエンコーダ検出（nvenc → libx264 フォールバック） ────
_HW_ENC = [None]  # キャッシュ: None=未検出
def _get_encoder():
    if _HW_ENC[0] is not None:
        return _HW_ENC[0]
    try:
        _t = subprocess.run(
            ['ffmpeg','-hide_banner','-f','lavfi','-i','color=black:s=64x64:r=1',
             '-t','0.1','-c:v','h264_nvenc','-f','null','-'],
            capture_output=True, timeout=5)
        if _t.returncode == 0:
            _HW_ENC[0] = 'h264_nvenc'
            print('  \U0001f680 GPU: h264_nvenc 検出')
            return _HW_ENC[0]
    except Exception:
        pass
    _HW_ENC[0] = 'libx264'
    return _HW_ENC[0]

# ── Pexels グローバルレートリミッター（全APIコールを4秒間隔に制限）───
_PEXELS_LAST_T = [0.0]
def _pexels_wait():
    gap = 4.0 - (time.time() - _PEXELS_LAST_T[0])  # 4s間隔でレート制限回避
    if gap > 0:
        time.sleep(gap)
    _PEXELS_LAST_T[0] = time.time()

# VOICEVOX定数（セル3で設定済みの値を継承、未実行時はデフォルト）
try: VV_AVAILABLE
except NameError: VV_AVAILABLE = False
try: VV_URL
except NameError: VV_URL = 'http://127.0.0.1:50021'
try: VV_SPEAKER
except NameError: VV_SPEAKER = 3
try: IMG_SWITCH_SEC
except NameError: IMG_SWITCH_SEC = 2.0
try: GOOGLE_API_KEY
except NameError: GOOGLE_API_KEY = ''
try: AUTO_UPLOAD
except NameError: AUTO_UPLOAD = False
try: UPLOAD_PRIVACY
except NameError: UPLOAD_PRIVACY = 'public'
try: SCHEDULE_POST
except NameError: SCHEDULE_POST = True
try: DRIVE_FOLDER_ID
except NameError: DRIVE_FOLDER_ID = ''
try: GEMINI_IMAGE_MODEL
except NameError: GEMINI_IMAGE_MODEL = ''
try: CHANNEL_MAP
except NameError: CHANNEL_MAP = []
try: POST_SLOTS_JST
except NameError: POST_SLOTS_JST = [8, 18, 21]
VIDEO_LEAD = 0.05

# ── テーマ別音声プロファイル ──────────────────────────────────────────
# VOICEVOXスピーカーID: 0=四国めたん/1=めたんあまあま/2=ずんだもんあまあま
#   3=ずんだもんノーマル/8=春日部つむぎ/10=雨晴はう/47=WhiteCUL/58=BLANC
_THEME_VOICE = [
    # キーワード → {vv_speaker, atempo, speed_scale, intonation}
    (['ダイエット','痩せ','体重','脂肪','カロリー'],
     {'vv_speaker': 3, 'atempo': 1.38, 'speed_scale': 1.08, 'intonation': 1.22}),
    (['美容','スキン','肌','化粧','コスメ','美白'],
     {'vv_speaker': 1, 'atempo': 1.22, 'speed_scale': 0.96, 'intonation': 1.05}),
    (['筋トレ','筋肉','トレーニング','ジム','プロテイン'],
     {'vv_speaker': 3, 'atempo': 1.45, 'speed_scale': 1.12, 'intonation': 1.28}),
    (['健康','習慣','生活','ウェルネス'],
     {'vv_speaker': 8, 'atempo': 1.30, 'speed_scale': 1.02, 'intonation': 1.12}),
    (['お金','マネー','節約','投資','副業','稼ぐ'],
     {'vv_speaker': 2, 'atempo': 1.32, 'speed_scale': 1.00, 'intonation': 1.10}),
    (['勉強','学習','英語','スキル','資格'],
     {'vv_speaker': 10, 'atempo': 1.28, 'speed_scale': 0.98, 'intonation': 1.08}),
]
_VOICE_DEFAULT = {'vv_speaker': 3, 'atempo': 1.35, 'speed_scale': 1.05, 'intonation': 1.15}

def get_theme_voice(theme):
    """テーマ文字列からVOICEVOXスピーカーID + gTTSテンポを返す"""
    for kws, cfg in _THEME_VOICE:
        if any(kw in theme for kw in kws):
            return dict(cfg)
    return dict(_VOICE_DEFAULT)

def load_history():
    """過去に生成した台本の先頭行リストを返す（プロンプトで除外指示に使用）"""
    if not HISTORY_FILE.exists():
        return []
    try:
        data = _json.loads(HISTORY_FILE.read_text(encoding='utf-8'))
        return [e.get('hook','') for e in data if e.get('hook')]
    except Exception:
        return []

def save_history(theme, angle, scene_texts):
    """生成完了後に台本フック行を履歴に追記"""
    try:
        data = []
        if HISTORY_FILE.exists():
            data = _json.loads(HISTORY_FILE.read_text(encoding='utf-8'))
        data.append({
            'theme': theme, 'angle': angle,
            'hook': scene_texts[0] if scene_texts else '',
            'lines': scene_texts[:6],  # 先頭6行を記録
        })
        # 最新100件だけ保持
        if len(data) > 100: data = data[-100:]
        HISTORY_FILE.write_text(_json.dumps(data, ensure_ascii=False, indent=1), encoding='utf-8')
    except Exception as _he:
        print(f'  ⚠ 履歴保存失敗: {_he}')
  # シーン2以降: 映像カットが音声より50ms先行
# 全シーン: 動画API優先で取得（VIDEO_SCENE_IDX廃止）
OUTPUT_DIR = Path('/content/output')
OUTPUT_DIR.mkdir(exist_ok=True)
HISTORY_FILE = OUTPUT_DIR / 'history.json'  # 台本履歴（重複防止）
completed = []
failed = []

def generate_visual_keywords(scene_texts, theme):
    """Step4: 確定したセリフからシーン固有の英語視覚キーワードを生成"""
    scenes_str = '\n'.join(f'シーン{i+1}: {t}' for i, t in enumerate(scene_texts))
    try:
        result = call_ai(
            f'各シーンのセリフを読んで、そのシーンの映像として最適な英語キーワードを生成してください。\n\n'
            f'【テーマ】{theme}\n'
            f'【セリフ一覧】\n{scenes_str}\n\n'
            f'【ルール】\n'
            f'① 各シーンで必ず違うキーワード（重複禁止）\n'
            f'② セリフの内容を映像化できる英語2〜3語の名詞フレーズ\n'
            f'③ NG: lifestyle / person / woman / man / healthy（単体）\n'
            f'④ OK: protein shake blender / steamed broccoli / morning alarm clock / '
            f'gym weights / fresh salad / glass of water / '
            f'running shoes / bedroom night / meal prep / coffee mug\n\n'
            f'【出力（英語のみ・厳守）】\n'
            f'scene1: （英語キーワード）\n'
            f'（全{len(scene_texts)}シーン分出力）',
            tokens=300
        )
        kws = []
        for line in result.strip().split('\n'):
            m = re.match(r'scene\d+[:\s]+(.+)', line.strip(), re.I)
            if m:
                kw = m.group(1).strip()
                if re.search(r'[\u3040-\u309F\u30A0-\u30FF\u4E00-\u9FFF]', kw):
                    kw = 'food close up'
                kws.append(kw)
        if len(kws) >= len(scene_texts):
            kws = [sanitize_visual_keyword(k, theme) for k in kws]
            print(f'  ✓ 視覚KW: {kws}')
            return kws[:len(scene_texts)]
    except Exception as _e:
        print(f'  ⚠ 視覚KW生成失敗: {_e}')
    return None

def rewrite_script_to_natural(scene_texts, theme, angle):
    """Step2: AI生成台本を日本人女性インフルエンサーの自然口語に完全リライト"""
    if not scene_texts:
        return scene_texts
    scenes_str = '\n'.join(f'シーン{i+1}: {t}' for i, t in enumerate(scene_texts))
    try:
        rewritten = call_ai(
            f'あなたは日本の人気女性YouTuberです。以下の台本を、スマホカメラに向かって'
            f'実際に話しているような100%自然な口語にリライトしてください。\n\n'
            f'【元の台本】\n{scenes_str}\n\n'
            f'【テーマ】{theme}  【タイトル】{angle}\n\n'
            f'【絶対に守るルール】\n'
            f'・「〜なんだ」「〜なの？」「〜するんだよ」「〜んです」など\n'
            f'  ロボット調・直訳調の語尾は完全に禁止\n'
            f'・語尾は「〜だよ！」「〜よね！」「〜してみて！」「〜のがポイント！」\n'
            f'  「〜じゃない？」など女性インフルエンサーが実際に使う語尾に統一\n'
            f'・①②③番号付きtipsは「〜する！」「〜を選ぶ！」の言い切り型に変換\n'
            f'・シーン1は「え、マジ？」「これ知ってた？」「実はね、」のような\n'
            f'  視聴者が思わず手を止める自然なフックで始める\n'
            f'・各セリフは10〜14文字の短いひとこと（長い文は絶対NG）\n'
            f'・ひらがな・カタカナ・漢字のみ（英語・記号・ハッシュタグ完全禁止）\n'
            f'・【コメント誘発】全シーンの中に1〜2箇所、視聴者が思わず突っ込むフレーズを入れる\n'
            f'  （例：「〜は絶対やるな！」「知らないと一生損する」「〜したら人生変わった」）\n\n'
            f'【出力フォーマット（厳守）】\n'
            f'セリフのみを出力。説明・コメント・空行は一切不要。\n'
            f'シーン1: （リライト後のセリフのみ）\n'
            f'シーン2: （リライト後のセリフのみ）\n'
            f'（元と同じシーン数で出力すること）',
            tokens=1024
        )
        # リライト結果をシーン単位でパース
        result = []
        for line in rewritten.strip().split('\n'):
            m = re.match(r'シーン\d+[:：]\s*(.+)', line.strip())
            if m:
                t = clean_text(m.group(1).strip())
                if t: result.append(t)
        # シーン数が一致しない場合は元テキストで補完
        while len(result) < len(scene_texts):
            result.append(scene_texts[len(result)])
        return result[:len(scene_texts)]
    except Exception as e:
        print(f'  ⚠ リライト失敗（元テキストにフォールバック）: {e}')
        return scene_texts

def make_one_video(idx, angle):
    vid_num = idx + 1
    safe_angle = re.sub(r'[\\/:*?"<>|\s]','_',angle)[:25]
    # BGM選択用にチャンネルラベルを設定
    try:
        import builtins as _bi
        _theme_now = globals().get('THEME', '')
        _cl_fn = globals().get('classify_theme_to_channel')
        _cur_lbl = _cl_fn(_theme_now) if _cl_fn else 'yaseru'
        globals()['_CURRENT_CHANNEL_LABEL'] = _cur_lbl
    except Exception:
        globals()['_CURRENT_CHANNEL_LABEL'] = 'yaseru'
    print(f'\n{"─"*50}')
    print(f'🎬 [{vid_num}/{VIDEO_COUNT}] {angle}')
    print(f'{"─"*50}')
    # ── チェックポイント: 既に完成済みならスキップ ─────────
    safe_theme_chk = re.sub(r'[\\/:*?"<>|\s]','_',THEME)[:15]
    safe_angle_chk = re.sub(r'[\\/:*?"<>|\s]','_',angle)[:25]
    OUT_CHK = OUTPUT_DIR/f'{vid_num:02d}_{safe_theme_chk}_{safe_angle_chk}.mp4'
    if OUT_CHK.exists() and OUT_CHK.stat().st_size > 50000:
        print(f'  ⏭️  スキップ（完成済み: {OUT_CHK.name}）')
        return str(OUT_CHK)

    TMP = Path(tempfile.mkdtemp(prefix=f'yt{vid_num}_'))
    MEDIA = TMP/'media'; MEDIA.mkdir()

    # ── テーマ別音声プロファイル決定 ─────────────────────────────────
    global _ACTIVE_VOICE_CFG
    _ACTIVE_VOICE_CFG = get_theme_voice(THEME)
    _vv_spk = _ACTIVE_VOICE_CFG['vv_speaker']
    print(f'  🔊 音声プロファイル: VV_SPEAKER={_vv_spk} / atempo={_ACTIVE_VOICE_CFG["atempo"]} (テーマ:{THEME})')

    # ── 台本生成: 1行=1テロップ形式（最大10文字・口語完全固定）──────
    print('  📝 台本生成中（1行1テロップ・10文字口語形式）...')
    _trend_ctx = globals().get('TREND_ANALYSIS', '')[:400]
    _trend_block = f'《最新トレンド》{_trend_ctx}\n\n' if _trend_ctx else ''
    _past_hooks = load_history()
    _hook_block = (
        f'【重複禁止（過去{len(_past_hooks)}本のフック）】\n'
        + '\n'.join(f'・{h}' for h in _past_hooks[-20:]) + '\n\n'
    ) if _past_hooks else ''
    _an_ctx = globals().get('_analytics_ctx', '')
    _an_block = f'《Analytics改善ヒント》{_an_ctx}\n\n' if _an_ctx else ''
    raw = call_ai(
        f'{_trend_block}'
        f'{_an_block}'
        f'{_hook_block}'
        f'YouTube Shortsのテロップ台本を作成してください。\n'
        f'【タイトル】{angle}\n【テーマ】{THEME}\n\n'
        f'【出力ルール（最重要・絶対厳守）】\n'
        f'・1行 = 1テロップ、最大１５文字\n'
        f'・15行以上 生成すること（15〜22行が最適）\n'
        f'・タイトルで「3つ」「5選」等を約束した場合は必ず全項目を完結させる\n'
        f'・番号付きリスト（①②③等）で各項目を明示してから内容を説明する\n'
        f'・最後の2行は必ず「これ保存して」→「すぐ試してね！」で締める\n'
        f'・シーン番号・説明文・記号（【】「」など）・空行 一切出力禁止\n'
        f'・テロップのみ縦１列で出力（余計な文字を一切混ぜない）\n\n'
        f'【金融・投資テーマの追加ルール（最重要）】\n'
        f'・具体的な商品名・数字・割合を含める（「いい感じに」等の抽象表現禁止）\n'
        f'・NISAコンテンツ: 株式/投資信託/ETFに限定。仮想通貨・ビットコイン・FXは絶対禁止\n'
        f'・タイトルが「〜3選」なら必ず①②③を全て台本内で説明する\n'
        f'・各銘柄・項目は2〜3行使って具体的に説明する（1行で済ませない）\n\n'
        f'【セリフルール（自然な日本語口語）】\n'
        f'・語尾: になっちゃうよ！/に変わるよ！/してみてね！/じゃない？/のがコツ！/やばい！/が大事！\n'
        f'・絶対禁止語尾: 〜になるんだよ/〜するんだよ/〜なんだよ/〜んです/〜でしょう/〜ましょう\n'
        f'  /〜目覚めさせちゃう！/〜なの？（解説調・翻訳調・ロボット語尾すべて禁止）\n'
        f'・1行目は必ず衝撃フック（え、マジ？/知ってた？/これやばい！ 等）\n'
        f'・ひらがな・カタカナ・漢字のみ（英語・記号・ハッシュタグ完全禁止）\n\n'
        f'【出力例（このフォーマット以外受け付けない）】\n'
        f'え、マジ？\n夜の3分で\n体が変わる！\n'
        f'食べながら\n痩せる秘密、\n知ってた？\n'
        f'夜8時以降は\nタンパク質を\n食べるだけ。\n'
        f'脂肪が燃える\n体に変わるよ！\n夜間の代謝が\n'
        f'爆上がりする！\n痩せた人は\nみんなこれ。\n'
        f'朝より夜が\n大チャンス！\n'
        f'これ保存して\nすぐ試してね！',
        tokens=600
    )
    # ── 1行1テロップとしてパース ──────────────────────────────────────
    _raw_lines = [l.strip() for l in raw.strip().split('\n') if l.strip()]
    _raw_lines = [l for l in _raw_lines
                  if not re.search(r'^[【\[（]|：|^scene|^シーン\d|^Step|^#|\d+[.。]$', l, re.I)]
    scene_texts = []
    for l in _raw_lines:
        t = re.sub(r'[#@\[\]（）(){}*_「」『』【】]', '', l).strip()
        t = re.sub(r'[！]{2,}', '！', t)
        if t:
            scene_texts.append(t[:15] if len(t) > 15 else t)
    # ── 語尾インフルエンサー自動置換（解説調→SNS口語） ──────────────────────────
    _ENDING_MAP = [
        (r'になるんだよ[！!]?', 'になっちゃうよ！'),
        (r'するんだよ[！!]?', 'してみてね！'),
        (r'できるんだよ[！!]?', 'できちゃうよ！'),
        (r'変わるんだよ[！!]?', 'に変わるよ！'),
        (r'燃えるんだよ[！!]?', 'が燃えるよ！'),
        (r'上がるんだよ[！!]?', '爆上がりするよ！'),
        (r'なんだよ[！!]?$', 'なんだよね！'),
    ]
    _replaced = []
    for _st in scene_texts:
        _s2 = _st
        for _pat, _rep in _ENDING_MAP:
            _s2 = re.sub(_pat, _rep, _s2)
        if _s2 != _st:
            print(f'  📝 語尾置換: {_st!r} → {_s2!r}')
        _replaced.append(_s2[:15] if len(_s2) > 15 else _s2)
    scene_texts = _replaced
    # ── 固定アウトロ保証 ─────────────────────────────────────────────
    _OUTRO = ['これ保存して', 'すぐ試してね！']
    while scene_texts and scene_texts[-1] in _OUTRO:
        scene_texts.pop()
    scene_texts.extend(_OUTRO)
    # ── 最小15行保証 ───────────────────────────────────────────────
    if len(scene_texts) < 15:
        print(f'  ⚠ {len(scene_texts)}行 → 20行に自動補完中...')
        _ch_pad = globals().get('_CURRENT_CHANNEL_LABEL', 'yaseru')
        if _ch_pad == 'okane':
            _pads = ['続けることが', '一番大事！', '毎月の積み立て', 'これが正解！',
                     'やってみよう！', '資産が増える！', '試してみて！', 'まず3ヶ月！',
                     '今日から始め', 'よう！', '絶対変われる', 'それだけ！']
        else:
            _pads = ['続けることが', '一番大事！', '毎日の積み重ね', 'これが正解！',
                     'やってみよう！', '変われるよ！', '試してみて！', 'まず1週間！',
                     '今日から始め', 'よう！', '絶対変われる', 'それだけ！']
        _pi = 0
        while len(scene_texts) < 20:
            scene_texts.insert(-2, _pads[_pi % len(_pads)])
            _pi += 1
    SCENE_COUNT = len(scene_texts)
    print(f'  ✓ Step1 台本生成完了（{SCENE_COUNT}行テロップ）')
    print(f'         フック: {scene_texts[0]!r} / アウトロ: {scene_texts[-2:]!r}')

    # ── Step2: 1行1テロップ形式のためリライト工程スキップ ───────────
    print('  ✅ Step2: 1行1テロップ形式 → 口語リライト工程スキップ')

    # ── Step3: 禁止語尾スキャン & 自動修正 ───────────────────
    print('  🔍 Step3: 禁止語尾スキャン中...')
    _NG_MAP = [('なの？','だよ！'), ('なんだよ','だよ！'), ('んです','だよ！'),
               ('でしょう','だよ！'), ('ましょう','しよう！'), ('むんだよ','だよ！'),
               ('目覚めさせちゃう！','やばい！')]
    _fc = 0
    for _si, _st in enumerate(scene_texts):
        for _ng, _rep in _NG_MAP:
            if _st.endswith(_ng):
                scene_texts[_si] = (_st[:-len(_ng)] + _rep)[:10]
                _fc += 1; break
    print(f'  {"✏ " + str(_fc) + "行の禁止語尾を自動修正" if _fc else "✓ 全行 禁止語尾なし"}')

    # ── 固定アウトロ二重チェック ───────────────────────────────────
    if scene_texts[-2:] != ['これ保存して', 'すぐ試してね！']:
        while scene_texts and scene_texts[-1] in ['これ保存して', 'すぐ試してね！']:
            scene_texts.pop()
        scene_texts.extend(['これ保存して', 'すぐ試してね！'])
        SCENE_COUNT = len(scene_texts)
    print(f'  🔄 固定アウトロ: {scene_texts[-2:]}')

    # ── Step4: 意味・映像完全同期 セマンティックマッチング ─────────────
    print('  🎯 Step4: セマンティック映像マッチング（1行:1クエリ固定）...')
    # 各テロップ行に意味一致クエリを1:1割り当て + BL Assert
    # AI一括生成優先（1 API call でシーン全体を文脈込みで生成）
    _ai_queries = ai_batch_visual_queries(scene_texts, THEME)
    if _ai_queries:
        print(f'  🤖 AI visual queries: {_ai_queries[:3]}...')
        line_queries = _ai_queries
    else:
        # フォールバック: 静的マップ
        line_queries = []
        for _lt in scene_texts:
            _q = get_semantic_query(_lt, THEME)
            try:
                assert_semantic_query(_q, _lt)
            except AssertionError as _ae:
                print(f'    ⚠ SemanticAssert: {_ae} → healthy food')
                _q = 'healthy food'
            line_queries.append(_q)
    print(f'  ✓ セマンティックKW: {line_queries[:5]}...(全{len(line_queries)}行)')

    # ── ハイブリッド画像生成: Gemini×3固定 + Pexels×残り ──────────────
    # 冒頭・中盤・ラスト の3カットのみGemini API呼び出し（月額コスト抑制）
    _total_cuts = len(line_queries)
    _gemini_indices = {
        0,                       # ①冒頭（1カット目）
        _total_cuts // 2,        # ②中盤（中央カット）
        _total_cuts - 1,         # ③ラスト（最終カット）
    }
    _gemini_cuts_display = sorted(i+1 for i in _gemini_indices)
    _GEMINI_CALL_COUNTER[0] = 0  # 各動画生成前にカウンターリセット
    _GRADIENT_USED[0] = False    # グラデBGフラグもリセット
    _USED_PEXELS_IDS.clear()        # Pexels重複排除セットをリセット
    _USED_PEXELS_VIDEO_IDS.clear()  # Pexels動画重複排除セットをリセット
    print(f'  🖼 ハイブリッド画像: Gemini×3（カット{_gemini_cuts_display}）+ Pexels×{_total_cuts-len(_gemini_indices)} / {IMG_SWITCH_SEC}秒切替')
    import random as _rand
    _diag_mode  = globals().get('VIDEO_FORMAT') == 'diagnosis'
    _diag_total = len(line_queries)
    _mascot_set = ({0, 1, _diag_total-2, _diag_total-1} if _diag_mode else set())

    # ── 動的ウィンドウグループ化（文字数ベース 2〜3.2秒）────────────
    _EST_CPS     = 5.5   # 日本語TTS 文字/秒
    _IMG_DUR_MIN = 2.0
    _IMG_DUR_MAX = 3.2
    _line_to_win = []    # 行インデックス → ウィンドウインデックス
    _win_repr_q  = {}    # ウィンドウインデックス → 代表クエリ
    if _diag_mode:
        for _ci in range(len(line_queries)):
            _line_to_win.append(_ci)
            _win_repr_q[_ci] = line_queries[_ci]
    else:
        _cur_win    = 0
        _accum      = 0.0
        _win_target = _rand.uniform(_IMG_DUR_MIN, _IMG_DUR_MAX)
        for _ci, _lq in enumerate(line_queries):
            _txt  = scene_texts[_ci] if _ci < len(scene_texts) else ''
            _est  = max(0.4, len(_txt) / _EST_CPS)
            if _accum > 0 and _accum + _est > _win_target:
                _cur_win += 1
                _win_target = max(_IMG_DUR_MIN, min(_IMG_DUR_MAX,
                    len(_txt) / _EST_CPS * 1.2 + _rand.uniform(0.2, 0.6)))
                _accum = 0.0
            if _cur_win not in _win_repr_q or len(_lq) > len(_win_repr_q.get(_cur_win,'')):
                _win_repr_q[_cur_win] = _lq
            _line_to_win.append(_cur_win)
            _accum += _est

    # ── 画像フェッチ（ウィンドウ代表クエリで1枚）────────────────────
    _total_wins  = len(_win_repr_q)
    _gemini_wins = {0, _total_wins // 2, _total_wins - 1}
    img_map = {}  # ウィンドウインデックス → Path
    _vid_map = {}  # ウィンドウインデックス → 動画クリップPath（Pexels動画）
    for _wi, _wq in _win_repr_q.items():
        _ifn    = MEDIA / f'img_{_wi+1:02d}.jpg'
        _rep_ci = next((i for i, w in enumerate(_line_to_win) if w == _wi), 0)
        _st     = scene_texts[_rep_ci] if _rep_ci < len(scene_texts) else ''
        if _diag_mode and _rep_ci in _mascot_set:
            fetch_mascot_image(_ifn, _rep_ci)
        elif _diag_mode:
            fetch_pexels_image_robust(_wq, _ifn, _wi, _st)
        elif _wi in _gemini_wins:
            fetch_gemini_image(_wq, _ifn, _wi)
        else:
            # 3カットおきにPexels動画クリップを試みる（非Geminiシーン）
            _try_video = (_wi % 3 == 2) or (_wi % 4 == 3)
            if _try_video:
                _vfn = MEDIA / f'pexels_vid_{_wi+1:02d}.mp4'
                _vresult = fetch_pexels_video(_wq, _vfn, _wi)
                if _vresult:
                    _vid_map[_wi] = _vfn
                    print(f'    Pexels動画クリップ取得: ウィンドウ{_wi+1}')
                else:
                    fetch_pexels_image_robust(_wq, _ifn, _wi, _st)
            else:
                fetch_pexels_image_robust(_wq, _ifn, _wi, _st)
        img_map[_wi] = _ifn
    _gemini_cuts_display = sorted(_wi + 1 for _wi in _gemini_wins)
    print(f'  🖼 画像: {len(img_map)}枚（Gemini×3 カット{_gemini_cuts_display} / 文字数ベース2〜3秒切替）')
    keywords = [_win_repr_q[_wi] for _wi in sorted(_gemini_wins) if _wi in _win_repr_q]

    # 診断系フォーマット: DIAGNOSIS_LINES でscene_textsを上書き
    _dl = globals().get('DIAGNOSIS_LINES', [])
    if globals().get('VIDEO_FORMAT') == 'diagnosis' and _dl:
        scene_texts  = list(_dl)
        line_queries = [l.lstrip('①②③④⑤⑥').strip() or THEME for l in _dl]
        print(f'  📋 診断スクリプト適用: {len(scene_texts)}行')

    # TTS用テキスト
    tts_texts = [tts_text_cleanse(t) for t in scene_texts]


    # ── ASMR tick: 診断項目切り替え時（②③④…）にチン音を追加 ──
    if globals().get('ASMR_TICK') and globals().get('VIDEO_FORMAT') == 'diagnosis':
        _tick_bytes = make_asmr_tick()
        _tick_path  = MEDIA / 'asmr_tick.wav'
        _tick_path.write_bytes(_tick_bytes)

    # scenes: 各行にウィンドウ画像を割り当て（同ウィンドウ内は同一画像）
    _fallback_img = list(img_map.values())[0] if img_map else MEDIA/'img_01.jpg'
    scenes = []
    for i, (scene_text, tts_t, _lq) in enumerate(zip(scene_texts, tts_texts, line_queries)):
        _wi = _line_to_win[i] if i < len(_line_to_win) else 0
        _has_pexels_vid = _wi in _vid_map
        scenes.append({'scene': i+1,
                       'image': img_map.get(_wi, _fallback_img),
                       'video': _vid_map[_wi] if _has_pexels_vid else MEDIA/f'vid_{i+1:02d}.mp4',
                       'is_video': _has_pexels_vid, 'keyword': _lq,
                       'text': scene_text, 'tts': tts_t})
    print(f'  ✓ ハイブリッド割当完了（{SCENE_COUNT}カット: Gemini×3 + Pexels×{SCENE_COUNT-3}）')

    # ── ループ構造: 最後のシーンのメディアを最初のシーンのものに完全一致させる ──
    # ── スタイル変換（静止画のみ） ────────────────────────────
    if IMAGE_STYLE in STYLES:
        fn_style = STYLES[IMAGE_STYLE]
        for s in scenes:
            if not s['is_video'] and s['image'].exists():
                img = cv2.imread(str(s['image']))
                if img is not None: cv2.imwrite(str(s['image']), fn_style(img))
        print('  ✓ スタイル変換完了')

    # ── 音声生成（フレーズ分割・句読点ごとに自然な間） ────────
    print('  🎙 音声生成中...')
    wavs = []
    # ── ElevenLabs: 全文一括生成（自然なイントネーション・数字読み改善）──
    _el_texts = [s.get('tts') or s['text'] or THEME for s in scenes]
    _el_result = elevenlabs_full_script(_el_texts, TMP) \
        if globals().get('ELEVENLABS_API_KEY', '').strip() else None
    _MIN_SCENE = 2.0  # テロップが読める最低秒数
    if _el_result:
        _el_splits, _el_per = _el_result
        for _si, (_s, _wv) in enumerate(zip(scenes, _el_splits)):
            _actual = get_wav_dur(_wv)
            if _actual < _MIN_SCENE:
                _pw = TMP / f'el_pad{_si:03d}.wav'
                try:
                    ff('-i', str(_wv), '-af',
                       f'apad=pad_dur={_MIN_SCENE - _actual:.3f}',
                       '-ar', '44100', '-ac', '1', str(_pw))
                    if _pw.exists(): _wv = _pw
                except Exception: pass
            _s['duration'] = max(get_wav_dur(_wv), _MIN_SCENE)
            wavs.append(_wv)
    else:
        # フォールバック: シーン別生成（ElevenLabs失敗 or キーなし）
        for i, s in enumerate(scenes):
            wav_out = make_tts_audio(s.get('tts') or s['text'] or THEME, TMP, f'tts{i}')
            if wav_out and wav_out.exists():
                audio_dur = get_wav_dur(wav_out)
                if audio_dur < _MIN_SCENE:
                    _pw2 = TMP / f'tts_pad{i:03d}.wav'
                    try:
                        ff('-i', str(wav_out), '-af',
                           f'apad=pad_dur={_MIN_SCENE - audio_dur:.3f}',
                           '-ar', '44100', '-ac', '1', str(_pw2))
                        if _pw2.exists(): wav_out = _pw2
                    except Exception: pass
                s['duration'] = max(get_wav_dur(wav_out), _MIN_SCENE)
                wavs.append(wav_out)
            else:
                s['duration'] = _MIN_SCENE
                print(f'  ⚠ 音声スキップ scene{i+1}')

    total_dur = sum(s['duration'] for s in scenes)
    # ── Duration Audit: 音声尺と映像尺を50ms以内に同期 ───────────────
    # ── Duration Audit: 実音声長と clip長を ±20ms 以内に厳格同期 ─────────
    print('  🔍 Duration Audit（厳格モード・VIDEO_LEAD廃止）...')
    _wavs_iter = iter(wavs)
    _drift_total = 0.0
    for _ai, _s in enumerate(scenes):
        _wf = next(_wavs_iter, None)
        if _wf and _wf.exists():
            _adur = get_wav_dur(_wf)          # WAVヘッダから実時間取得
            _exp  = max(_adur, 0.5)            # 実音声長のみ（VIDEO_LEAD完全禁止）
            _drift = abs(_s['duration'] - _exp)
            if _drift > 0.02:                  # 20ms超えたら即修正
                print(f'    ⚠ scene{_ai+1}: 音声={_adur:.3f}s clip={_s["duration"]:.3f}s ズレ={_drift*1000:.0f}ms → 強制修正')
                _s['duration'] = _exp
                _drift_total += _drift
    if _drift_total == 0:
        print(f'  ✅ Duration Audit: 全{len(scenes)}シーン ±20ms以内')
    else:
        total_dur = sum(s['duration'] for s in scenes)
        print(f'  ✅ Duration Audit: {_drift_total*1000:.0f}ms修正完了 → total={total_dur:.3f}s')
    cap_dur = total_dur  # 0msアウトロ: 最後の音声終了と同時に完全切断（無限ループ最適化）
    print(f'  ✓ 音声完了 合計{total_dur:.3f}秒 → 出力{cap_dur:.3f}秒（0msアウトロ確定）')

    # ── 動画クリップ生成 ──────────────────────────────────────
    print('  🎬 クリップ生成中（3秒超えシーン自動ジャンプカット分割）...')
    SPLIT_THRESHOLD = 2.5  # 2.5秒超えシーンは前半(通常) + 後半(静的1.25xセンターズーム)
    XFADE_DUR = 0.30     # クロスフェード時間(秒): 0.3秒でなめらかなディゾルブ
    clips = []
    clip_durs = []       # xfade計算用: 各クリップの生成時間
    for i_c, s in enumerate(scenes):
        d = s['duration']
        has_vid = s['is_video'] and s['video'].exists()

        def _mk(dur, out_path, kb_idx, boost=1.0, static_zoom=None):
            """ソース種別に応じたクリップ生成（static_zoom=1.25で後半センターズーム）"""
            if has_vid:
                try:
                    make_clip_from_video(s['video'], dur, out_path, idx=kb_idx, boost=boost)
                    return
                except Exception as e:
                    print(f'  ⚠ 動画変換失敗→静止画: {e}')
            make_clip(s['image'], dur, out_path, idx=kb_idx, boost=boost, static_zoom=static_zoom)

        if d > SPLIT_THRESHOLD:
            # 3秒超え: 前半（通常）+ 後半（1.2倍ズームアップ）で疑似ジャンプカット
            half = d / 2
            out1 = TMP/f"c{s['scene']:02d}a.mp4"
            out2 = TMP/f"c{s['scene']:02d}b.mp4"
            _mk(half + XFADE_DUR, out1, i_c, boost=1.0)
            _mk(half + XFADE_DUR, out2, i_c + 1, static_zoom=1.25)
            clips.extend([out1, out2])
            clip_durs.extend([half + XFADE_DUR, half + XFADE_DUR])
            print(f"  ✂ scene{s['scene']} ジャンプカット分割 ({half:.1f}s × 2 / 後半1.25xセンターズーム)")
        else:
            out = TMP/f"c{s['scene']:02d}.mp4"
            _mk(d + XFADE_DUR, out, i_c, boost=1.0)
            clips.append(out)
            clip_durs.append(d + XFADE_DUR)

    # クロスフェードでクリップを結合（ハードカット → なめらかなディゾルブ）
    print(f'  🎞 xfadeトランジション結合中（{len(clips)}クリップ / {XFADE_DUR}s dissolve）...')
    mg = TMP/'m.mp4'
    _ch_trans = globals().get('_CURRENT_CHANNEL_LABEL', 'yaseru')
    _scene_trans = get_scene_transitions(len(clips), _ch_trans)
    _trans_types = [t for t, _ in _scene_trans]
    print(f'  🎬 トランジション: {_trans_types[:6]}...' if len(_trans_types) > 6 else f'  🎬 トランジション: {_trans_types}')
    concat_with_xfade(clips, clip_durs, mg, XFADE_DUR, _scene_trans)

    # ── 字幕（黄色・大きめ） ─────────────────────────────────
    ah = (
        f"[Script Info]\nPlayResX:{W}\nPlayResY:{H}\nScriptType:v4.00+\n\n"
        f"[V4+ Styles]\n"
        f"Format:Name,Fontname,Fontsize,PrimaryColour,SecondaryColour,OutlineColour,BackColour,"
        f"Bold,Italic,Underline,StrikeOut,ScaleX,ScaleY,Spacing,Angle,BorderStyle,Outline,Shadow,"
        f"Alignment,MarginL,MarginR,MarginV,Encoding\n"
        f"Style:Default,Noto Sans CJK JP,75,&H0000FFFF,&H000000FF,&H00000000,&HA0000000,"
        f"-1,0,0,0,100,100,2,0,3,12,0,2,{int(W*0.10)},{int(W*0.10)},{int(H*0.28)},1\n"
        f"Style:Hook,Noto Sans CJK JP,76,&H00FFFFFF,&H000000FF,&H00000000,&H000000FF,"
        f"-1,0,0,0,100,100,0,0,3,20,0,8,60,60,{int(H*0.05)},1\n\n"
        f"[Events]\nFormat:Layer,Start,End,Style,Name,MarginL,MarginR,MarginV,Effect,Text\n"
    )
    ev = []; t = 0.0
    # ── 冒頭フックテロップ（0〜2秒・Hook スタイル・赤背景）───────────
    if '】' in angle and '【' in angle:
        _hl1 = angle[:angle.index('】')+1]; _hl2 = angle[angle.index('】')+1:].strip()
        _htxt = _hl1 + r'\N' + _hl2 if _hl2 else _hl1
    elif len(angle) > 10:
        _m = len(angle)//2; _htxt = angle[:_m] + r'\N' + angle[_m:]
    else:
        _htxt = angle
    # 行溢れ防止: 最長行が12文字超ならフォントサイズを自動縮小
    # Hook style: 76pt × 1文字≈76px, 有効幅960px (1080-60×2margins) → 最大12文字
    _hook_lines = [l for l in _htxt.split(r'\N') if l]
    _max_hook_len = max(len(l) for l in _hook_lines) if _hook_lines else 1
    if _max_hook_len > 12:
        _hook_fs = max(40, int(960 // _max_hook_len))
        _htxt = r'{\fs' + str(_hook_fs) + r'}' + _htxt
    _hend = min(2.0, scenes[0]['duration'] if scenes else 2.0)
    ev.append(f"Dialogue:0,{at(0)},{at(_hend)},Hook,,0,0,0,,{_htxt}")
    # 字幕: 最大12文字・1行・1〜2秒カード切り替え（Y=62%固定）
    # ── LipSync Assert: 音声・字幕・映像 3点完全同期を事前検証 ───────────
    print('  🔒 LipSync Assert検証中...')
    assert len(scenes) == len(wavs), \
        f'シーン数({len(scenes)})とWAV数({len(wavs)})が不一致 → テキストと音声がズレる'
    _lip_ok = True
    _t_acc = 0.0
    for _li, (_ls2, _lw) in enumerate(zip(scenes, wavs)):
        _la = get_wav_dur(_lw) if _lw and _lw.exists() else 0.0
        _ld = abs(_ls2['duration'] - max(_la, 0.5)) * 1000  # ms
        if _ld > 30:  # 30ms超えはNG
            print(f'    ❌ scene{_li+1}: 音声={_la:.3f}s clip={_ls2["duration"]:.3f}s ズレ={_ld:.0f}ms')
            _ls2['duration'] = max(_la, 0.5)  # 強制修正
            _lip_ok = False
        _t_acc += _ls2['duration']
    if _lip_ok:
        print(f'  ✅ LipSync Assert: 全{len(scenes)}シーン同期OK / 総尺{_t_acc:.3f}s')
    else:
        print(f'  ⚠  LipSync Assert: 違反を強制修正済 → 総尺{_t_acc:.3f}s')

    # ── 字幕タイムライン: 実音声積算タイムスタンプで構築（理論値禁止）──────
    # t = 実音声の積算時刻 (= 映像カット点 = 音声開始点 = 字幕表示開始点)
    # VIDEO_LEAD を使った「音声より字幕が先走る」バグを根絶
    SUB_CX = W // 2
    SUB_CY = int(H * 0.62)
    for idx_s, s in enumerate(scenes):
        # 実音声長をscene durationとして字幕スパンを算出
        _sub_dur = s['duration']          # 実音声長（Duration Audit確認済み）
        _wav_dur = get_wav_dur(wavs[idx_s]) if idx_s < len(wavs) and wavs[idx_s].exists() else _sub_dur
        # 字幕は音声発声期間と完全1:1同期
        c_s_abs = t                        # 音声発声開始 = 字幕表示開始
        c_e_abs = t + _wav_dur - 1.0/FPS  # 音声終了1フレーム前に消す
        # テロップは16文字以下の行のみ表示（短い文で理解しやすい）
        _sub_raw = re.sub(r'#\S+', '', clean_subtitle_text(s['text'])).strip()
        if not _sub_raw or len(_sub_raw) > 16:
            t += s['duration']; continue  # 長いテキストは音声のみ
        cards = make_sub_cards(s['text'], _wav_dur)
        cards = validate_sub_cards(cards, s['text'], _wav_dur)
        for card_text, c_s_off, c_e_off in cards:
            c_s = c_s_abs + c_s_off
            c_e = min(c_s_abs + c_e_off, c_e_abs)
            ev.append(
                f"Dialogue:0,{at(c_s)},{at(max(c_s+0.05, c_e))},Default,,0,0,0,,"
                f"{{\\an5\\pos({SUB_CX},{SUB_CY})\\fs75}}{card_text}"
            )
        t += s['duration']  # 実音声長で積算（ズレ蓄積なし）

    sub = TMP/'s.ass'
    sub.write_text(ah+'\n'.join(ev), encoding='utf-8')
    se = str(sub).replace('\\','/').replace(':','\\:')

    # ── ナレーション音声を結合 ────────────────────────────────
    comb = TMP/'vc.wav'; vaac = TMP/'v.aac'
    # シーン1以降の先頭に VIDEO_LEAD 分の無音を挿入（映像50ms先行同期）
    # VIDEO_LEAD無音挿入廃止: 全WAVを直接結合（音声・字幕・映像の完全同期）
    _aparts = [wf for wf in wavs if wf and wf.exists()]
    if not _aparts:
        # 全シーン音声なし（フォールバック）
        wio.write(str(comb), 44100, np.zeros(int(44100*total_dur), dtype=np.float32))
    elif len(_aparts) == 1:
        import shutil as _sh; _sh.copy(str(_aparts[0]), str(comb))
    else:
        lf = TMP/'vl.txt'
        lf.write_text('\n'.join(f"file '{p}'" for p in _aparts))
        ff('-f','concat','-safe','0','-i',str(lf),'-c','copy',str(comb))
    # silenceremove廃止: VIDEO_LEAD除去後は人工無音ゼロなので不要。
    # 適用すると音声長が変わりタイムライン同期が破壊されるためスキップ。
    print('  ✅ silenceremove スキップ（1行1テロップ形式・直接結合 → 人工無音なし）')
    ff('-i',str(comb),'-c:a','aac','-ar','44100',str(vaac))

    # ── BGM生成・ノイズフロア生成・最終合成 ─────────────────────
    bgm_path = TMP/'bgm.wav'
    _ch_lbl_bgm = globals().get('_CURRENT_CHANNEL_LABEL', 'yaseru')
    if not load_bgm_from_drive(cap_dur + 2, bgm_path, _ch_lbl_bgm):
        wio.write(str(bgm_path), 44100, generate_bgm(cap_dur + 2, channel=_ch_lbl_bgm))
    # Lo-Fiノイズフロア: ぶつ切り無音を聴覚的に隠蔽（約-30dB）
    noise_path = TMP/'noise.wav'
    _nsr = 44100; _nn = int(_nsr * (cap_dur + 2))
    _rng = np.random.default_rng(42)
    _raw = _rng.standard_normal(_nn).astype(np.float32)
    # 簡易ローパス（Lo-Fi感: 2kHz以下）: 16サンプル移動平均
    _k = np.ones(16, dtype=np.float32) / 16
    _smooth = np.convolve(_raw, _k, mode='same')
    _smooth = (_smooth / (np.abs(_smooth).max() + 1e-9) * 0.03).astype(np.float32)
    wio.write(str(noise_path), _nsr, _smooth)
    print('  🎵 BGM + ノイズフロア完了')

    # ── シームレスループ: TTS合計尺で0フレーム余白なくカット ─────────
    # total_dur = 各シーンのTTS時間の合計（VIDEO_LEAD 込み）
    # vaac のエンコーダ遅延（≈0.023s）は -t で切り落とす → 末尾無音ゼロ
    cap_dur = round(min(max(total_dur, 15.0), 58.0), 2)  # Shorts上限58s
    print(f'  ✂️  シームレスループ: total_dur={total_dur:.3f}s → cap_dur={cap_dur:.2f}s')

    safe_theme_f = re.sub(r'[\\/:*?"<>|\s]','_',THEME)[:15]
    OUT = OUTPUT_DIR/f'{vid_num:02d}_{safe_theme_f}_{safe_angle}.mp4'

    # ── LAYER4: レンダリング前 全仕様自動監査 & 自己修正 ──────────────────
    ev, _audit_ok = pre_render_audit(scenes, ev, cap_dur, keywords, THEME)
    sub.write_text(ah+'\n'.join(ev), encoding='utf-8')  # 修正後ASS再書き込み

    ff('-i',str(mg),'-i',str(vaac),'-i',str(bgm_path),'-i',str(noise_path),
       '-filter_complex',
       '[1:a]volume=1.0[narr];'
       '[2:a]volume=0.65[bgm];'
       '[3:a]volume=1.0[nz];'
       '[narr][bgm][nz]amix=inputs=3:duration=first:normalize=0[aout]',
       '-vf',f'ass={se}',
       '-map','0:v','-map','[aout]',
       '-c:v','libx264','-preset','fast','-crf','20',
       '-c:a','aac','-b:a','192k',
       '-pix_fmt','yuv420p','-movflags','+faststart',
       '-t',str(cap_dur),str(OUT))

    if not OUT.exists() or OUT.stat().st_size < 10000:
        raise RuntimeError(f'動画生成失敗: {OUT}')

    mb = OUT.stat().st_size/1_048_576
    print(f'  ✅ 完成！ {OUT.name} ({mb:.1f}MB, {cap_dur:.3f}秒)')
    save_history(THEME, angle, scene_texts)
    # グラデBG混入チェック: 使用があった場合はアップロードスキップ対象に登録
    if _GRADIENT_USED[0]:
        print(f'  ⚠ グラデBG混入あり → Cell6アップロードをスキップします（ローカル保存は完了）')
        _GRADIENT_SKIP_PATHS.add(str(OUT))
    return str(OUT)

# ── Pexels APIキー事前確認 ──────────────────────────────────────────────
print('🔑 Pexels APIキー確認中...')
_pk_test = ''.join(c for c in (PEXELS_API_KEY or '') if 0x21 <= ord(c) <= 0x7E)
if not _pk_test:
    print('  ❌ PEXELS_API_KEY が空です！セル1でキーを設定してください。')
else:
    try:
        _tr = _req.get('https://api.pexels.com/v1/search',
                       headers={'Authorization': _pk_test},
                       params={'query': 'food', 'per_page': 1}, timeout=10)
        _rem = _tr.headers.get('X-Ratelimit-Remaining', '不明')
        if _tr.status_code == 200:
            print(f'  ✅ Pexels接続OK  key={_pk_test[:6]}...  残リクエスト={_rem}')
            _PEXELS_LAST_T[0] = time.time()
        elif _tr.status_code == 401:
            print(f'  ❌ Pexels: APIキーが無効 (401) → 全シーンにグラデーション背景')
        elif _tr.status_code == 429:
            print(f'  ⚠ Pexels: レート制限中 (429) → 30秒待機')
            time.sleep(30)
        else:
            print(f'  ⚠ Pexels: HTTP {_tr.status_code}')
    except Exception as _te:
        print(f'  ⚠ Pexels接続テスト失敗: {_te}')
print()

# ── 再開チェック: 既存完成動画を確認 ──────────────────────
import traceback
_safe_th = re.sub(r'[\\/:*?"<>|\s]','_',THEME)[:15]
_already = [f for f in OUTPUT_DIR.glob(f'??_{_safe_th}_*.mp4') if f.stat().st_size > 50000]
if _already:
    print(f'📂 完成済み動画 {len(_already)}本 を検出 → 該当番号はスキップします')
    for _f in sorted(_already): print(f'   ✓ {_f.name}')
else:
    print('📂 完成済みなし → 全本新規生成')

start_all = time.time()

# ── Analytics フィードバック読み込み（2回目以降に自動改善）────────
_analytics_ctx = ''
try:
    from googleapiclient.discovery import build as _drv_build
    import google.auth as _gauth_c5
    _creds_c5, _ = _gauth_c5.default(scopes=['https://www.googleapis.com/auth/drive.readonly'])
    _drv_c5 = _drv_build('drive', 'v3', credentials=_creds_c5)
    _dfid = (DRIVE_FOLDER_ID or '').strip()
    if _dfid:
        _af = _drv_c5.files().list(
            q=f"name='analytics_insights.json' and '{_dfid}' in parents and trashed=false",
            fields='files(id)'
        ).execute().get('files', [])
        if _af:
            import io as _io_c5
            _req_c5 = _drv_c5.files().get_media(fileId=_af[0]['id'])
            _buf_c5 = _io_c5.BytesIO()
            from googleapiclient.http import MediaIoBaseDownload as _MIBD
            _dl = _MIBD(_buf_c5, _req_c5)
            while True:
                _, _done = _dl.next_chunk()
                if _done: break
            _ai_data = _json.loads(_buf_c5.getvalue().decode('utf-8'))
            _analytics_ctx = _ai_data.get('summary', '')[:300]
            print(f'  📊 Analytics読み込み完了: {_analytics_ctx[:60]}...')
except Exception as _ac5:
    pass  # 初回実行時 or Drive未設定時はスキップ

# ── 全テーマを一括処理 ──────────────────────────────────────────────
_all_themes = globals().get('THEMES', [globals().get('THEME', 'ダイエット')])
_global_idx = 0
for _ti, _cur_theme in enumerate(_all_themes):
    globals()['THEME'] = _cur_theme
    globals()['_CURRENT_CHANNEL_LABEL'] = classify_theme_to_channel(_cur_theme)
    _ch_name = 'やせる習慣図鑑' if globals()['_CURRENT_CHANNEL_LABEL']=='yaseru' else 'お金の教科書'
    print(f'\n{"="*55}')
    print(f'🎯 テーマ {_ti+1}/{len(_all_themes)}: 「{_cur_theme}」→ {_ch_name}')
    print(f'{"="*55}')
    if _ti > 0:
        # 2テーマ目以降はCell4の関数でANGLES再生成
        if 'generate_angles_for_theme' in globals():
            generate_angles_for_theme(_cur_theme)
        else:
            print('  ⚠ generate_angles_for_theme未定義 → Cell4を再実行してください')
            continue
    for idx, angle in enumerate(ANGLES):
        try:
            out_path = make_one_video(_global_idx, angle)
            completed.append((_global_idx+1, angle, out_path))
        except Exception as e:
            print(f'\n  ❌ エラー: {traceback.format_exc()[-500:]}')
            failed.append((_global_idx+1, angle, str(e)))
        _global_idx += 1

_gradient_count = len(_GRADIENT_SKIP_PATHS)
_upload_count = len(completed) - _gradient_count
if _gradient_count:
    print(f'  ⚠ グラデBG混入: {_gradient_count}本 → アップロードスキップ対象')
    print(f'  ✅ アップロード可能: {_upload_count}本')

elapsed = (time.time()-start_all)/60
print(f'\n{"="*50}')
print(f'🎉 完了！ 成功:{len(completed)}本 / 失敗:{len(failed)}本 / {elapsed:.1f}分')
print(f'{"="*50}')


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  client_secrets.json を直接貼り付け（初回のみ）     ║
# ║  ファイルアップロードが難しい場合はこのセルを使う   ║
# ╚══════════════════════════════════════════════════════╝
#
# 手順:
#   1. Google Cloud Console からダウンロードした JSON ファイルを
#      メモ帳/テキストエディタで開く
#   2. 全選択（Ctrl+A）→ コピー（Ctrl+C）
#   3. 下の _secrets_json = ''' の後ろに貼り付けて実行

import json as _j, os as _os

_secrets_json = '''
← ここにJSONの中身を貼り付ける（{"installed":{"client_id":... から始まるテキスト）
'''

_secrets_json = _secrets_json.strip()
if not _secrets_json or _secrets_json.startswith('←'):
    print('⚠ JSONの中身を貼り付けてから実行してください')
else:
    try:
        _j.loads(_secrets_json)  # JSON構文チェック
        with open('/content/client_secrets.json', 'w') as _sf:
            _sf.write(_secrets_json)
        print('✅ /content/client_secrets.json を作成しました')
        print('   次はセル6（YouTube自動投稿）を実行してください')
    except _j.JSONDecodeError as _je:
        print(f'❌ JSON形式エラー: {_je}')
        print('   メモ帳で開いた内容をそのまま貼り付けてください')


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  セル6: YouTube自動投稿（セル5の後に実行）                           ║
# ║                                                                      ║
# ║  初回のみ:                                                            ║
# ║  1. Google Cloud Console → YouTube Data API v3 を有効化             ║
# ║  2. Colabシークレットに CLIENT_SECRETS_JSON を登録                   ║
# ║  3. 下記の認証URLを開いて承認 → レスポンスURLを貼り付け               ║
# ╚══════════════════════════════════════════════════════════════════════╝

import os, json, pickle, urllib.parse, time, re, random, requests as _req6
from datetime import datetime, timezone, timedelta
from pathlib import Path as _P6

os.environ['OAUTHLIB_INSECURE_TRANSPORT'] = '1'

# ── 定数 ──────────────────────────────────────────────────────────────
_SCOPES6 = [
    'https://www.googleapis.com/auth/youtube.upload',
    'https://www.googleapis.com/auth/youtube.force-ssl',
]

# ⚙ チャンネルID設定（オプション: 空のままでOK。自動検出されます）
# Google Cloud Console → YouTube Data API → チャンネルID から確認
# 例: YASERU_CHANNEL_ID = 'UCxxxxxxxxxxxxxxxxx'
#     OKANE_CHANNEL_ID  = 'UCxxxxxxxxxxxxxxxxx'
try: YASERU_CHANNEL_ID
except NameError: YASERU_CHANNEL_ID = ''
try: OKANE_CHANNEL_ID
except NameError: OKANE_CHANNEL_ID = ''
_CHANNEL_ID_MAP6 = {}
if YASERU_CHANNEL_ID.strip(): _CHANNEL_ID_MAP6['yaseru'] = YASERU_CHANNEL_ID.strip()
if OKANE_CHANNEL_ID.strip():  _CHANNEL_ID_MAP6['okane']  = OKANE_CHANNEL_ID.strip()
_JST6   = timezone(timedelta(hours=9))
_YT_BASE = 'https://www.googleapis.com/youtube/v3'
_YT_UPLOAD = 'https://www.googleapis.com/upload/youtube/v3/videos'
# ════════════════════════════════════════════════════════
# 🔍 事前チェック（セル実行前に全依存を検証）
# ════════════════════════════════════════════════════════
def _preflight_check():
    errors = []
    warnings = []

    # 1. client_secrets.json 存在確認
    from pathlib import Path as _pfP
    _sec_paths = [_pfP('/content/client_secrets.json'),
                  _pfP('/content/drive/MyDrive/client_secrets.json')]
    if not any(p.exists() for p in _sec_paths):
        errors.append('client_secrets.json が見つかりません（セル6を先に実行）')

    # 2. ElevenLabs APIキー確認
    _el_key = globals().get('ELEVENLABS_API_KEY','').strip()
    if not _el_key:
        warnings.append('ELEVENLABS_API_KEY 未設定 → gTTS にフォールバック')

    # 3. 完了済み動画の確認
    _comp = globals().get('completed', [])
    if not _comp:
        warnings.append('completed リストが空です（セル5を先に実行してください）')

    # 4. ffprobe / ffmpeg の存在確認
    import subprocess as _pfs
    for _cmd in ['ffmpeg', 'ffprobe']:
        try:
            _pfs.run([_cmd, '-version'], capture_output=True, timeout=5)
        except Exception:
            errors.append(f'{_cmd} が見つかりません（セル2でインストール済みか確認）')

    # 結果表示
    if errors:
        for e in errors: print(f'  ❌ {e}')
        raise RuntimeError(f'事前チェック失敗（{len(errors)}件）: 上記を解決してから再実行')
    for w in warnings: print(f'  ⚠ {w}')
    print('  ✅ 事前チェック通過')

_preflight_check()

# ── Google Drive 自動マウント（トークンが Drive に保存されているため必須）──
try:
    if not _P6('/content/drive/MyDrive').exists():
        from google.colab import drive as _gd
        _gd.mount('/content/drive', force_remount=False)
        print('  ✅ Google Drive マウント完了')
    else:
        print('  ✅ Google Drive 接続済み')
except Exception as _dm_e:
    print(f'  ⚠ Drive マウント失敗: {_dm_e}（トークンがローカルに保存される場合があります）')
_TOKEN_DIR6 = _P6('/content/drive/MyDrive') if _P6('/content/drive/MyDrive').exists() else _P6('/content')
if str(_TOKEN_DIR6) == '/content':
    print('  ⚠ Drive未接続のためトークンはセッション内のみ有効です')
else:
    print(f'  💾 トークン保存先: {_TOKEN_DIR6}')
_PENDING_F6 = _TOKEN_DIR6 / 'yt_pending_comments.json'

# ── セル1未実行フォールバック ─────────────────────────────────────────
try: CLAUDE_API_KEY
except NameError: CLAUDE_API_KEY = ''
try: AUTO_UPLOAD
except NameError: AUTO_UPLOAD = True
try: SCHEDULE_POST
except NameError: SCHEDULE_POST = True
try: UPLOAD_PRIVACY
except NameError: UPLOAD_PRIVACY = 'public'
try: POST_WINDOWS
except NameError: POST_WINDOWS = [(8,0,30),(12,0,30),(18,0,30)]
try: completed
except NameError: completed = []
try: classify_theme_to_channel
except NameError:
    def classify_theme_to_channel(t): return 'yaseru'

# ── client_secrets.json ロード ────────────────────────────────────────
def _load_secrets6():
    for p in ['/content/client_secrets.json',
              '/content/drive/MyDrive/client_secrets_youtube.json']:
        if os.path.exists(p):
            return json.load(open(p))
    s = ''
    try:
        from google.colab import userdata as _ud6
        s = (_ud6.get('CLIENT_SECRETS_JSON') or '').strip()
    except Exception: pass
    if s:
        with open('/content/client_secrets.json','w') as _f: _f.write(s)
        return json.loads(s)
    raise FileNotFoundError('client_secrets.json が見つかりません。セル1を再実行してください。')

# ── トークンクラス ─────────────────────────────────────────────────────
class _Token6:
    def __init__(self, access_token, refresh_token, client_id, client_secret, expires_at=0):
        self.access_token  = access_token
        self.refresh_token = refresh_token
        self.client_id     = client_id
        self.client_secret = client_secret
        self.expires_at    = expires_at
    @property
    def expired(self): return time.time() >= self.expires_at - 60
    def refresh(self):
        r = _req6.post('https://oauth2.googleapis.com/token', data={
            'refresh_token': self.refresh_token,
            'client_id':     self.client_id,
            'client_secret': self.client_secret,
            'grant_type':    'refresh_token',
        }, timeout=30)
        d = r.json()
        if 'error' in d: raise ValueError(f'トークン更新失敗: {d}')
        self.access_token = d['access_token']
        self.expires_at   = time.time() + d.get('expires_in', 3600)

def _save_tok6(tok, label):
    p = _TOKEN_DIR6 / f'yt_token_{label}.pickle'
    with open(p,'wb') as f: pickle.dump(tok, f)
    print(f'  💾 トークン保存: {p}')

def _load_tok6(label):
    p = _TOKEN_DIR6 / f'yt_token_{label}.pickle'
    if not p.exists(): return None
    with open(p,'rb') as f: tok = pickle.load(f)
    if not hasattr(tok, 'access_token') or not hasattr(tok, 'refresh_token'): p.unlink(); return None
    if tok.expired and tok.refresh_token:
        try: tok.refresh(); _save_tok6(tok, label); print(f'  🔄 {label}: トークン自動更新')
        except Exception as e: print(f'  ⚠ {label}: 更新失敗 → 再認証要 ({e})'); p.unlink(); return None
    return tok

def _auth_url6(label):
    sec = _load_secrets6()['installed']
    return 'https://accounts.google.com/o/oauth2/auth?' + urllib.parse.urlencode({
        'client_id':     sec['client_id'],
        'redirect_uri':  'http://localhost',
        'response_type': 'code',
        'scope':         ' '.join(_SCOPES6),
        'access_type':   'offline',
        'prompt':        'consent',
        'state':         f'yt_{label}',
    })

def _exchange6(resp_url, label):
    sec = _load_secrets6()['installed']
    code = urllib.parse.parse_qs(urllib.parse.urlparse(resp_url).query).get('code',[''])[0]
    if not code: raise ValueError(f'URLにcodeが見つかりません: {resp_url[:100]}')
    r = _req6.post('https://oauth2.googleapis.com/token', data={
        'code': code, 'client_id': sec['client_id'],
        'client_secret': sec['client_secret'],
        'redirect_uri': 'http://localhost', 'grant_type': 'authorization_code',
    }, timeout=30)
    d = r.json()
    if 'error' in d: raise ValueError(f'トークン交換失敗: {d}')
    tok = _Token6(d['access_token'], d.get('refresh_token',''),
                  sec['client_id'], sec['client_secret'],
                  time.time() + d.get('expires_in', 3600))
    _save_tok6(tok, label); return tok

# ── YouTube軽量クライアント ────────────────────────────────────────────
class _YT6:
    def __init__(self, token): self.token = token.access_token
    def _h(self): return {'Authorization': f'Bearer {self.token}', 'Content-Type': 'application/json'}
    def channel_name(self):
        r = _req6.get(f'{_YT_BASE}/channels', params={'part':'snippet','mine':'true'},
                      headers=self._h(), timeout=15)
        r.raise_for_status()
        items = r.json().get('items',[])
        return items[0]['snippet']['title'] if items else '不明'
    def channel_id(self):
        r = _req6.get(f'{_YT_BASE}/channels', params={'part':'snippet','mine':'true'},
                      headers=self._h(), timeout=15)
        r.raise_for_status()
        items = r.json().get('items',[])
        return items[0]['id'] if items else None
    def upload(self, path, body):
        sz = os.path.getsize(path)
        h = {**self._h(), 'X-Upload-Content-Type':'video/mp4',
             'X-Upload-Content-Length':str(sz)}
        r = _req6.post(_YT_UPLOAD, params={'uploadType':'resumable','part':'snippet,status'},
                       headers=h, data=json.dumps(body), timeout=30)
        r.raise_for_status()
        url = r.headers['Location']
        done, CHUNK = 0, 8*1024*1024
        with open(path,'rb') as f:
            while done < sz:
                chunk = f.read(CHUNK)
                end = done + len(chunk) - 1
                r2 = _req6.put(url, data=chunk, timeout=300,
                               headers={'Authorization':f'Bearer {self.token}',
                                        'Content-Range':f'bytes {done}-{end}/{sz}',
                                        'Content-Type':'video/mp4'})
                if r2.status_code in (200,201): return r2.json()['id']
                elif r2.status_code == 308:
                    done = end + 1
                    print(f'    {int(done/sz*100)}%', end='\r')
                else: r2.raise_for_status()
    def comment(self, vid_id, text):
        r = _req6.post(f'{_YT_BASE}/commentThreads', params={'part':'snippet'},
                       headers=self._h(), timeout=15,
                       data=json.dumps({'snippet':{'videoId':vid_id,
                           'topLevelComment':{'snippet':{'textOriginal':text}}}}))
        r.raise_for_status()
    def status(self, vid_id):
        r = _req6.get(f'{_YT_BASE}/videos', params={'part':'status','id':vid_id},
                      headers=self._h(), timeout=15)
        r.raise_for_status()
        items = r.json().get('items',[])
        return items[0]['status']['privacyStatus'] if items else None

# ── ユーティリティ ────────────────────────────────────────────────────
def _random_slot6(idx=0):
    now = datetime.now(_JST6)
    slots = []
    for d in range(14):
        for h, ms, me in POST_WINDOWS:
            m = random.randint(ms, me)
            s = random.randint(0, 59)
            t = (now + timedelta(days=d)).replace(hour=h, minute=m, second=s, microsecond=0)
            if t > now + timedelta(hours=2): slots.append(t)
    slots.sort()
    return slots[idx % len(slots)].astimezone(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')

def _gen_meta6(theme, channel_label, scene_lines):
    if not CLAUDE_API_KEY: return {'title':f'{theme} #Shorts','description':f'#{theme}','tags':[theme],'first_comment':'保存してね！'}
    preview = '\n'.join(scene_lines[:8]) if scene_lines else ''
    ch_name = 'やせる習慣図鑑' if channel_label=='yaseru' else 'お金の教科書'
    try:
        r = _req6.post('https://api.anthropic.com/v1/messages',
            headers={'x-api-key':CLAUDE_API_KEY,'anthropic-version':'2023-06-01','content-type':'application/json'},
            json={'model':'claude-haiku-4-5-20251001','max_tokens':600,
                  'messages':[{'role':'user','content':
                    f'YouTube Shortsのメタデータ。テーマ:{theme} チャンネル:{ch_name}\n台本冒頭:\n{preview}\n\n'
                    f'タイトルルール:\n'
                    f'・台本の内容を具体的に反映した独自タイトル（20〜30文字）\n'
                    f'・「〜キロやってた人」「〜キロ落とした人」「知らないと損」は使わない\n'
                    f'・疑問形・逆説・具体行動・発見 など多様な形式を使う\n'
                    f'・末尾に #Shorts を付ける\n\n'
                    '以下のJSON形式のみ出力(説明不要):\n'
                    '{"title":"20〜30文字 #Shorts末尾","description":"150文字以内+ハッシュタグ6-8個","tags":["タグ1","タグ2"],"first_comment":"保存やシェアを促す40文字以内"}'}]},
            timeout=30)
        raw = r.json()['content'][0]['text']
        m = re.search(r'\{.*\}', raw, re.DOTALL)
        if m: return json.loads(m.group())
    except Exception as e: print(f'  ⚠ メタデータ生成失敗: {e}')
    return {'title':f'{theme} の習慣 #Shorts','description':f'#{theme} #shorts','tags':[theme,'shorts'],'first_comment':'保存してね！'}

def _load_pending6():
    if not _PENDING_F6.exists(): return []
    try: return json.loads(_PENDING_F6.read_text())
    except: return []

def _save_pending6(data):
    _PENDING_F6.write_text(json.dumps(data, ensure_ascii=False, indent=2))

def _flush_pending6(yt):
    data = _load_pending6()
    if not data: return
    remaining = []
    for item in data:
        try:
            if yt.status(item['video_id']) == 'public':
                yt.comment(item['video_id'], item['comment'])
                print(f'  ✅ 保留コメント投稿: {item["video_id"]}')
            else: remaining.append(item)
        except Exception as e:
            print(f'  ⚠ 保留コメント確認失敗: {e}')
            remaining.append(item)
    _save_pending6(remaining)
    if len(data) != len(remaining):
        print(f'  📋 保留コメント: {len(data)-len(remaining)}件投稿, {len(remaining)}件残り')

# ── 認証チェック + 必要なら認証ガイド ────────────────────────────────
print('='*60)
print('🔐 YouTube認証確認')
print('='*60)

_LABELS6 = {'yaseru': 'やせる習慣図鑑', 'okane': 'お金の教科書'}
_auth_map6 = {}   # label → _YT6 client
_need_auth6 = []  # 認証が必要なlabel

# 完了した動画からどのチャンネルが必要か確認
_needed_labels6 = set()
try:
    _secrets6 = _load_secrets6()
    _themes6 = globals().get('THEMES', [globals().get('THEME', '')])
    for _t6 in _themes6:
        _needed_labels6.add(classify_theme_to_channel(_t6))
    if completed and not _needed_labels6: _needed_labels6 = set(_LABELS6.keys())
    if not completed: _needed_labels6 = set(_LABELS6.keys())
except Exception as _se6:
    print(f'  ⚠ client_secrets.json読み込み失敗: {_se6}')
    _needed_labels6 = set()

# チャンネルID / チャンネル名 → ラベル のマッピング（IDが優先）
_CH_NAME_MAP6 = {
    'やせる習慣図鑑': 'yaseru',
    'お金の教科書':   'okane',
}
# セル1のチャンネルID設定（Colabシークレットから読み込まれる）
try: _y_cid = YASERU_CHANNEL_ID.strip()
except NameError: _y_cid = ''
try: _o_cid = OKANE_CHANNEL_ID.strip()
except NameError: _o_cid = ''
_CH_ID_MAP6 = {}  # channel_id → label
if _y_cid: _CH_ID_MAP6[_y_cid] = 'yaseru'
if _o_cid: _CH_ID_MAP6[_o_cid] = 'okane'
if _CH_ID_MAP6:
    print(f'  🔑 チャンネルID照合: yaseru={_y_cid[:8]}... okane={_o_cid[:8]}...')

for _lbl6 in _needed_labels6:
    _tok6 = _load_tok6(_lbl6)
    if _tok6:
        _yt6 = _YT6(_tok6)
        try:
            _ch_n6 = _yt6.channel_name()
            _ch_id6 = _yt6.channel_id()
            # ① チャンネルIDで照合（最優先・確実）
            if _CH_ID_MAP6 and _ch_id6 in _CH_ID_MAP6:
                _actual_lbl6 = _CH_ID_MAP6[_ch_id6]
            else:
                # ② チャンネル名でフォールバック
                _actual_lbl6 = _CH_NAME_MAP6.get(_ch_n6, _lbl6)

            if _actual_lbl6 != _lbl6:
                print(f'  ℹ {_lbl6}トークン → 実チャンネル: {_ch_n6} (ID:{_ch_id6}) → {_actual_lbl6}として登録')
            else:
                print(f'  ✅ {_ch_n6} (ID:{_ch_id6}): 認証済み')
            _auth_map6[_actual_lbl6] = _yt6
            _flush_pending6(_yt6)
            # _actual_lbl6 で登録したので _lbl6 のチェックは不要
            # （チャンネルが入れ替わっていても認証済みとみなす）
        except Exception as _ae6:
            print(f'  ⚠ {_LABELS6[_lbl6]}: トークンエラー ({_ae6}) → 再認証要')
            _need_auth6.append(_lbl6)
    else:
        _need_auth6.append(_lbl6)

# 登録済みチャンネルを表示
print()
for _k6, _v6 in _auth_map6.items():
    print(f'  📺 {_LABELS6.get(_k6,_k6)}: 投稿準備完了')
if not _auth_map6:
    print('  ❌ 認証済みチャンネルなし')

# 未認証チャンネルの認証ガイド
for _lbl6 in _need_auth6:
    _ch_n6 = _LABELS6[_lbl6]
    print()
    print(f'【{_ch_n6} の認証手順】')
    print(f'  ⓵ YouTubeで「{_ch_n6}」に切り替える')
    print(f'  ⓶ 以下のURLをコピーしてブラウザで開く:')
    try:
        print(f'\n  {_auth_url6(_lbl6)}\n')
    except Exception as _ue6:
        print(f'  ❌ URL生成失敗: {_ue6}'); continue
    print(f'  ⓷ Googleでログインページが開く')
    print(f'  ⓸ ⚠ 重要: アカウント選択画面で「{_ch_n6}」ブランドアカウントを選ぶ')
    print(f'     （メインのGmailアカウントではなく、YouTubeチャンネルのアカウントを選択）')
    print(f'  ⓹ 権限を承認する')
    print(f'  ⓺ リダイレクトされた http://localhost/?code=... URLをコピー')
    print(f'  ⓻ 新しいセルを追加して以下を貼り付け、URLだけ書き換えて実行:')
    print()
    print(f'  _resp = "ここにhttp://localhost/?code=...のURLを貼り付け"')
    print(f'  _exchange6(_resp, "{_lbl6}")')
    print(f'  print("→ セル7を再実行してください")')
    print()
    print(f'  ※ブランドアカウントが選択肢に出ない場合:')
    print(f'    YouTube(youtube.com)でチャンネルを「{_ch_n6}」に切り替えてから')
    print(f'    上記URLをブラウザで開いてください')
    print()

# ── アップロード ──────────────────────────────────────────────────────
if not completed:
    print()
    print('❌ 投稿する動画がありません。セル5を先に実行してください。')
elif not AUTO_UPLOAD:
    print()
    print('⏭ AUTO_UPLOAD=False → アップロードスキップ')
elif not _auth_map6:
    print()
    print('⚠ 認証済みチャンネルがありません。上記の手順で認証してください。')
else:
    print()
    print('='*60)
    print('⬆️  YouTube自動投稿開始')
    print('='*60)
    _results6 = []
    _slot6 = 0
    _skip_set6 = globals().get('_GRADIENT_SKIP_PATHS', set())

    for _ci6, _ca6, _cp6 in completed:
        if str(_cp6) in _skip_set6:
            print(f'  ⏭ グラデBG混入スキップ: {_P6(_cp6).name}'); continue

        # テーマからチャンネル判定
        _theme6 = globals().get('THEME', '')
        _lbl6   = classify_theme_to_channel(_theme6)
        _yt6    = _auth_map6.get(_lbl6)
        if not _yt6:
            # 必要なチャンネルが未認証の場合は詳細な案内を表示
            _has_labels = list(_auth_map6.keys())
            print(f'  ⚠ {_LABELS6.get(_lbl6,"?")} 未認証 → スキップ')
            print(f'     (認証済み: {[_LABELS6.get(k,k) for k in _has_labels]})')
            print(f'     → 上記の手順で {_LABELS6.get(_lbl6,_lbl6)} のブランドアカウントを認証してください')
            print(f'     → OAuth画面でブランドアカウントを選択するのが重要です')
            continue

        _ch_name6 = _LABELS6[_lbl6]
        print(f'\n  [{_ci6}] {_P6(_cp6).name} → {_ch_name6}')

        # スクリプトライン取得
        _lines6 = []
        try:
            _hf = _P6('/content/drive/MyDrive') / 'yt_history.json'
            if not _hf.exists(): _hf = _P6('/content/yt_history.json')
            if _hf.exists():
                _hd = json.loads(_hf.read_text())
                if _hd: _lines6 = _hd[min(_ci6-1,len(_hd)-1)].get('lines',[])
        except Exception as _hist_e:
            print(f'  ⚠ 履歴ファイル読み込みスキップ: {_hist_e}')

        # メタデータ生成
        _meta6 = _gen_meta6(_theme6, _lbl6, _lines6)
        print(f'  タイトル: {_meta6["title"]}')

        # 予約時刻
        _sched6 = _random_slot6(_slot6) if SCHEDULE_POST else None
        if _sched6:
            _jst6 = datetime.fromisoformat(_sched6.replace('Z','+00:00')).astimezone(_JST6)
            print(f'  📅 予約(JST): {_jst6.strftime("%m/%d %H:%M")}')
            _slot6 += 1

        # ボディ
        _body6 = {
            'snippet': {'title':_meta6['title'],'description':_meta6['description'],
                        'tags':_meta6.get('tags',[]),'categoryId':'26' if _lbl6=='yaseru' else '22',
                        'defaultLanguage':'ja'},
            'status':  {'privacyStatus':'private' if _sched6 else UPLOAD_PRIVACY,
                        'selfDeclaredMadeForKids':False,'containsSyntheticMedia':True},
        }
        if _sched6: _body6['status']['publishAt'] = _sched6

        try:
            _vid6 = _yt6.upload(_cp6, _body6)
            _url6 = f'https://youtube.com/watch?v={_vid6}'
            print(f'  ✅ 投稿完了: {_url6}')
            _results6.append(_url6)
            # コメント
            if _sched6:
                _pend6 = _load_pending6()
                _pend6.append({'video_id':_vid6,'comment':_meta6['first_comment']})
                _save_pending6(_pend6)
                print(f'  📋 コメント保留（公開後に自動投稿）')
            else:
                try: _yt6.comment(_vid6, _meta6['first_comment']); print('  💬 コメント投稿完了')
                except: print('  ⚠ コメント投稿失敗（後で手動投稿）')
        except Exception as _upe6:
            print(f'  ❌ アップロード失敗: {_upe6}')

    print()
    print('='*60)
    print(f'🎉 完了: {len(_results6)}/{len(completed)}本を投稿')
    for _u6 in _results6: print(f'  {_u6}')
    print('='*60)
